In [1]:
import pandas as pd
import numpy as np
import math
from __future__ import annotations
import os
from dataclasses import dataclass
from typing import Dict, List, Optional
import yfinance as yf
import base64
from io import BytesIO
from datetime import datetime
import matplotlib.pyplot as plt
import argparse
from zoneinfo import ZoneInfo

### Creating trading book

In [2]:
positions = pd.DataFrame(
    [
        # RATES DESK
        {
            "desk": "Rates",
            "ticker": "IEF",
            "weight": 0.35,
            "risk_type": "Rates",
            "risk_factor": "USD_curve",
            "bucket": "USD_5Y",
            "description": "Intermediate US Treasury exposure"
        },
        {
            "desk": "Rates",
            "ticker": "TLT",
            "weight": -0.15,
            "risk_type": "Rates",
            "risk_factor": "USD_curve",
            "bucket": "USD_20Y",
            "description": "Long-end US Treasury exposure"
        },

        # FX DESK
        {
            "desk": "FX",
            "ticker": "EURUSD",
            "weight": 0.25,
            "risk_type": "FX",
            "risk_factor": "EURUSD",
            "bucket": "G10_FX",
            "description": "EURUSD spot exposure"
        },
        {
            "desk": "FX",
            "ticker": "USDJPY",
            "weight": -0.10,
            "risk_type": "FX",
            "risk_factor": "USDJPY",
            "bucket": "G10_FX",
            "description": "USDJPY spot exposure"
        },

        # CREDIT DESK
        {
            "desk": "Credit",
            "ticker": "LQD",
            "weight": 0.30,
            "risk_type": "Credit",
            "risk_factor": "IG_spread",
            "bucket": "Investment_Grade",
            "description": "IG corporate bond exposure"
        },
        {
            "desk": "Credit",
            "ticker": "HYG",
            "weight": -0.20,
            "risk_type": "Credit",
            "risk_factor": "HY_spread",
            "bucket": "High_Yield",
            "description": "HY corporate bond exposure"
        },
    ]
)

## SANITY CHECKS ##

# Ensure multiple desks
assert positions["desk"].nunique() >= 3

# Ensure long & short exposure exists
assert (positions["weight"] > 0).any() and (positions["weight"] < 0).any()

## EXPOSURE GOVERNANCE CONTROLS ##

gross_exposure = positions["weight"].abs().sum()
net_exposure = positions["weight"].sum()

# Gross exposure cap (risk appetite style control)
MAX_GROSS = 2.5
assert gross_exposure <= MAX_GROSS, "Gross exposure exceeds governance threshold"

# Desk concentration cap
desk_gross = positions.groupby("desk")["weight"].apply(lambda x: x.abs().sum())
MAX_DESK_GROSS = 1.5
assert (desk_gross <= MAX_DESK_GROSS).all(), "Desk concentration too high"

## OUTPUT SUMMARY ##

print("=== TRADING BOOK SUMMARY ===")
print(positions)
print("\nNet exposure:", round(net_exposure, 4))
print("Gross exposure:", round(gross_exposure, 4))
print("\nDesk gross exposures:")
print(desk_gross)

# Save
positions.to_csv("positions_stress_ready.csv", index=False)
print("\nWrote positions_stress_ready.csv")

=== TRADING BOOK SUMMARY ===
     desk  ticker  weight risk_type risk_factor            bucket  \
0   Rates     IEF    0.35     Rates   USD_curve            USD_5Y   
1   Rates     TLT   -0.15     Rates   USD_curve           USD_20Y   
2      FX  EURUSD    0.25        FX      EURUSD            G10_FX   
3      FX  USDJPY   -0.10        FX      USDJPY            G10_FX   
4  Credit     LQD    0.30    Credit   IG_spread  Investment_Grade   
5  Credit     HYG   -0.20    Credit   HY_spread        High_Yield   

                         description  
0  Intermediate US Treasury exposure  
1      Long-end US Treasury exposure  
2               EURUSD spot exposure  
3               USDJPY spot exposure  
4         IG corporate bond exposure  
5         HY corporate bond exposure  

Net exposure: 0.45
Gross exposure: 1.35

Desk gross exposures:
desk
Credit    0.50
FX        0.35
Rates     0.50
Name: weight, dtype: float64

Wrote positions_stress_ready.csv


### Data collection

In [3]:
@dataclass
class IngestConfig:
    positions_path: str = "positions_stress_ready.csv"
    out_dir: str = "data"
    start: str = "2010-01-01"
    end: Optional[str] = None
    price_field: str = "Adj Close"
    stale_days_threshold: int = 3
    extreme_return_threshold: float = 0.10  # 10%
    missing_pct_threshold: float = 0.02     # 2%


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def ticker_mapping(user_tickers: List[str]) -> Dict[str, str]:
    """
    Map internal tickers to yfinance symbols.
    - ETFs: unchanged
    - FX (yfinance convention):
        EURUSD -> EURUSD=X
        USDJPY -> JPY=X
    """
    mapping = {t: t for t in user_tickers}
    fx_map = {
        "EURUSD": "EURUSD=X",
        "USDJPY": "JPY=X",
    }
    for k, v in fx_map.items():
        if k in mapping:
            mapping[k] = v
    return mapping


def download_prices(symbols: List[str], start: str, end: Optional[str], price_field: str) -> pd.DataFrame:
    df = yf.download(
        tickers=symbols,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="column",
        actions=False,
        threads=True,
    )

    if isinstance(df.columns, pd.MultiIndex):
        if price_field not in df.columns.get_level_values(0):
            raise ValueError(f"Missing '{price_field}' in downloaded fields: {df.columns.levels[0].tolist()}")
        prices = df[price_field].copy()
    else:
        if price_field not in df.columns:
            raise ValueError(f"Missing '{price_field}' in columns: {df.columns.tolist()}")
        prices = df[[price_field]].rename(columns={price_field: symbols[0]}).copy()

    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()
    return prices


def compute_returns(prices: pd.DataFrame) -> pd.DataFrame:
    rets = prices.pct_change(fill_method=None)
    rets = rets.dropna(how="all")
    return rets


def longest_stale_run(prices_col: pd.Series) -> int:
    s = prices_col.dropna()
    if s.empty:
        return 0
    diffs = s.diff()
    is_stale = diffs.eq(0.0).fillna(False).astype(int)

    longest = 0
    cur = 0
    for v in is_stale.values:
        if v == 1:
            cur += 1
            longest = max(longest, cur)
        else:
            cur = 0
    return int(longest)


def data_quality_checks(prices: pd.DataFrame, returns: pd.DataFrame, cfg: IngestConfig) -> pd.DataFrame:
    rows = []
    for t in prices.columns:
        missing_pct = float(prices[t].isna().mean())
        stale_run = longest_stale_run(prices[t])

        r = returns[t].dropna() if t in returns.columns else pd.Series(dtype=float)
        extreme_cnt = int((r.abs() > cfg.extreme_return_threshold).sum())

        zero_return_cnt = int((r == 0).sum())

        rows.append(
            {
                "ticker": t,
                "missing_pct": missing_pct,
                "longest_stale_run_days": stale_run,
                "extreme_return_count": extreme_cnt,
                "zero_return_count": zero_return_cnt,
                "extreme_return_threshold": cfg.extreme_return_threshold,
                "stale_days_threshold": cfg.stale_days_threshold,
                "missing_pct_threshold": cfg.missing_pct_threshold,
                "flag_missing": missing_pct > cfg.missing_pct_threshold,
                "flag_stale": stale_run >= cfg.stale_days_threshold,
                "flag_extreme": extreme_cnt > 5,
                "flag_many_zeros": zero_return_cnt > 10,
            }
        )

    out = pd.DataFrame(rows)
    return out.sort_values(
        ["flag_missing", "flag_stale", "flag_extreme", "flag_many_zeros"],
        ascending=False
    )


def compute_true_asof_date(prices: pd.DataFrame) -> pd.Timestamp:
    """
    true_asof_date: last date where ALL tickers were observed (non-NaN).
    This mirrors the 'complete risk factor set' requirement.
    """
    observed = prices.notna()
    complete_observed = observed.all(axis=1)
    return prices.index[complete_observed].max() if complete_observed.any() else pd.NaT

## Main ##

cfg = IngestConfig()
ensure_dir(cfg.out_dir)

positions = pd.read_csv(cfg.positions_path)

# Ensuring required columns exist
required_cols = {"desk", "ticker", "weight", "risk_type", "risk_factor", "bucket"}
missing = required_cols - set(positions.columns)
if missing:
    raise ValueError(f"positions file missing columns: {sorted(missing)}")

# Save ticker metadata
ticker_meta = (
    positions[["ticker", "desk", "risk_type", "risk_factor", "bucket"]]
    .drop_duplicates()
    .sort_values(["desk", "risk_type", "risk_factor", "ticker"])
)

user_tickers = ticker_meta["ticker"].unique().tolist()

mapping = ticker_mapping(user_tickers)
symbols = [mapping[t] for t in user_tickers]

prices_raw = download_prices(symbols, cfg.start, cfg.end, cfg.price_field)

# Rename columns back to internal tickers
inv_map = {v: k for k, v in mapping.items()}
prices = prices_raw.rename(columns=inv_map)

# Reference last date present in the downloaded frame
file_last_date = prices.index.max() if len(prices.index) else pd.NaT

# drop rows where all prices missing
prices = prices.dropna(how="all")

returns = compute_returns(prices)

dq = data_quality_checks(prices, returns, cfg)

true_asof_date = compute_true_asof_date(prices)

stale_any = prices.diff().eq(0.0).fillna(False).any(axis=1)
complete_any = prices.notna().all(axis=1)
safe_dates = complete_any & (~stale_any)
safe_asof_date = prices.index[safe_dates].max() if safe_dates.any() else pd.NaT

# outputs
prices_path = os.path.join(cfg.out_dir, "prices.csv")
returns_path = os.path.join(cfg.out_dir, "returns.csv")
dq_path = os.path.join(cfg.out_dir, "data_quality.csv")
meta_path = os.path.join(cfg.out_dir, "asof_metadata.csv")
tmeta_path = os.path.join(cfg.out_dir, "ticker_metadata.csv")

prices.to_csv(prices_path, index_label="date")
returns.to_csv(returns_path, index_label="date")
dq.to_csv(dq_path, index=False)
ticker_meta.to_csv(tmeta_path, index=False)

meta = pd.DataFrame(
    [
        {
            "true_asof_date": "" if pd.isna(true_asof_date) else pd.Timestamp(true_asof_date).strftime("%Y-%m-%d"),
            "safe_asof_date": "" if pd.isna(safe_asof_date) else pd.Timestamp(safe_asof_date).strftime("%Y-%m-%d"),
            "file_last_date": "" if pd.isna(file_last_date) else pd.Timestamp(file_last_date).strftime("%Y-%m-%d"),
            "start": cfg.start,
            "end": "" if cfg.end is None else cfg.end,
            "positions_path": cfg.positions_path,
            "price_field": cfg.price_field,
        }
    ]
)
meta.to_csv(meta_path, index=False)

print(f"Wrote {prices_path}  (rows={len(prices)}, cols={prices.shape[1]})")
print(f"Wrote {returns_path} (rows={len(returns)}, cols={returns.shape[1]})")
print(f"Wrote {dq_path}")
print(f"Wrote {tmeta_path}")
print(f"Wrote {meta_path}")

print("\nAs-of metadata:")
print(meta.to_string(index=False))
print("\nData quality summary:")
print(dq.to_string(index=False))

Wrote data/prices.csv  (rows=4277, cols=6)
Wrote data/returns.csv (rows=4276, cols=6)
Wrote data/data_quality.csv
Wrote data/ticker_metadata.csv
Wrote data/asof_metadata.csv

As-of metadata:
true_asof_date safe_asof_date file_last_date      start end             positions_path price_field
    2026-05-29     2026-05-29     2026-05-29 2010-01-01     positions_stress_ready.csv   Adj Close

Data quality summary:
ticker  missing_pct  longest_stale_run_days  extreme_return_count  zero_return_count  extreme_return_threshold  stale_days_threshold  missing_pct_threshold  flag_missing  flag_stale  flag_extreme  flag_many_zeros
   HYG     0.035305                       3                     0                 59                       0.1                     3                   0.02          True        True         False             True
   IEF     0.035305                       2                     0                 32                       0.1                     3                   0.02       

### Desk and Total Profit & Loss

In [4]:
POSITIONS_PATH = "positions_stress_ready.csv"
RETURNS_PATH = os.path.join("data", "returns.csv")
OUT_DIR = "data"

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

ensure_dir(OUT_DIR)

# Load positions
pos = pd.read_csv(POSITIONS_PATH)

required_cols = {"desk", "ticker", "weight", "risk_type", "risk_factor", "bucket"}
missing = required_cols - set(pos.columns)
if missing:
    raise ValueError(f"{POSITIONS_PATH} missing columns: {missing}")

# Load returns (wide): date index, tickers as columns
rets = pd.read_csv(RETURNS_PATH, parse_dates=["date"]).set_index("date").sort_index()

# Check that all tickers in positions exist in returns
tickers = pos["ticker"].unique().tolist()
missing_tickers = [t for t in tickers if t not in rets.columns]
if missing_tickers:
    raise ValueError(
        f"These tickers are in {POSITIONS_PATH} but not in {RETURNS_PATH}: {missing_tickers}\n"
        f"Available columns: {list(rets.columns)}"
    )

rets_sub = rets[tickers]
complete_dates = rets_sub.notna().all(axis=1)
rets_sub = rets_sub.loc[complete_dates]

if rets_sub.empty:
    raise ValueError("No dates with complete returns across all tickers. Check ingestion/data quality output.")

# Aggregate weights by ticker 
w = pos.groupby("ticker", as_index=True)["weight"].sum()

# Instrument-level P&L proxy: pnl = weight * return
pnl_instrument_wide = rets_sub.mul(w, axis=1)

# Convert to long format with metadata attached
meta_cols = ["desk", "ticker", "risk_type", "risk_factor", "bucket"]
pnl_long = (
    pnl_instrument_wide.stack()
    .rename("pnl")
    .reset_index()
    .rename(columns={"level_1": "ticker"})
    .merge(pos[meta_cols].drop_duplicates(), on="ticker", how="left")
    .sort_values(["date", "desk", "ticker"])
)

# Desk P&L and total P&L
pnl_desk = (
    pnl_long.groupby(["date", "desk"], as_index=False)["pnl"]
    .sum()
    .sort_values(["date", "desk"])
)

pnl_total = (
    pnl_long.groupby(["date"], as_index=False)["pnl"]
    .sum()
    .rename(columns={"pnl": "pnl_total"})
    .sort_values(["date"])
)

# factor-level P&L 
pnl_factor = (
    pnl_long.groupby(["date", "risk_factor"], as_index=False)["pnl"]
    .sum()
    .sort_values(["date", "risk_factor"])
)

# outputs
pnl_long.to_csv(os.path.join(OUT_DIR, "pnl_instrument.csv"), index=False)
pnl_desk.to_csv(os.path.join(OUT_DIR, "pnl_desk.csv"), index=False)
pnl_total.to_csv(os.path.join(OUT_DIR, "pnl_total.csv"), index=False)
pnl_factor.to_csv(os.path.join(OUT_DIR, "pnl_risk_factor.csv"), index=False)

print(f"Wrote {os.path.join(OUT_DIR, 'pnl_instrument.csv')} (rows={len(pnl_long)})")
print(f"Wrote {os.path.join(OUT_DIR, 'pnl_desk.csv')}      (rows={len(pnl_desk)})")
print(f"Wrote {os.path.join(OUT_DIR, 'pnl_total.csv')}     (rows={len(pnl_total)})")
print(f"Wrote {os.path.join(OUT_DIR, 'pnl_risk_factor.csv')} (rows={len(pnl_factor)})")

# Sanity checks
gross = float(pos["weight"].abs().sum())
net = float(pos["weight"].sum())
print("\nSanity checks:")
print("Date range:", pnl_total["date"].min().date(), "to", pnl_total["date"].max().date())
print("Net exposure (sum weights):", round(net, 4))
print("Gross exposure (sum abs weights):", round(gross, 4))
print("Total P&L mean:", float(pnl_total["pnl_total"].mean()))
print("Total P&L stdev:", float(pnl_total["pnl_total"].std()))
print("Complete dates used:", int(complete_dates.sum()), "/", int(len(complete_dates)))

Wrote data/pnl_instrument.csv (rows=23772)
Wrote data/pnl_desk.csv      (rows=11886)
Wrote data/pnl_total.csv     (rows=3962)
Wrote data/pnl_risk_factor.csv (rows=19810)

Sanity checks:
Date range: 2010-01-05 to 2026-05-29
Net exposure (sum weights): 0.45
Gross exposure (sum abs weights): 1.35
Total P&L mean: 5.62799277456016e-06
Total P&L stdev: 0.002232244079722143
Complete dates used: 3962 / 4276


### Historical VaR and Diversification Benefit

In [5]:
PNL_DESK_PATH = os.path.join("data", "pnl_desk.csv")
PNL_TOTAL_PATH = os.path.join("data", "pnl_total.csv")
OUT_DIR = "data"

WINDOW = 250
ALPHA_VAR = 0.99     # VaR 99%
ALPHA_ES  = 0.975    # ES 97.5% (FRTB-style)


def historical_var(series: pd.Series, alpha: float) -> float:
    """Historical VaR as a positive number (loss)."""
    q = np.quantile(series, 1 - alpha)
    return -q


def historical_es(series: pd.Series, alpha: float) -> float:
    """Historical ES as a positive number (loss)."""
    q = np.quantile(series, 1 - alpha)
    tail = series[series <= q]
    if len(tail) == 0:
        return np.nan
    return -tail.mean()


# Load P&L
pnl_desk = pd.read_csv(PNL_DESK_PATH, parse_dates=["date"])
pnl_total = pd.read_csv(PNL_TOTAL_PATH, parse_dates=["date"])

# Desk-level VaR + ES 
desk_rows = []

for desk, g in pnl_desk.groupby("desk"):
    g = g.sort_values("date").set_index("date")

    var_series = g["pnl"].rolling(WINDOW, min_periods=WINDOW).apply(
        lambda x: historical_var(x, ALPHA_VAR), raw=False
    )
    es_series = g["pnl"].rolling(WINDOW, min_periods=WINDOW).apply(
        lambda x: historical_es(x, ALPHA_ES), raw=False
    )

    tmp = pd.DataFrame(
        {
            "date": var_series.index,
            "desk": desk,
            "var_99": var_series.values,
            "es_975": es_series.values,
        }
    ).dropna(subset=["var_99", "es_975"])

    desk_rows.append(tmp)

risk_desk = pd.concat(desk_rows, ignore_index=True)

# Total VaR + ES
pnl_total = pnl_total.sort_values("date").set_index("date")

var_total_series = pnl_total["pnl_total"].rolling(WINDOW, min_periods=WINDOW).apply(
    lambda x: historical_var(x, ALPHA_VAR), raw=False
)
es_total_series = pnl_total["pnl_total"].rolling(WINDOW, min_periods=WINDOW).apply(
    lambda x: historical_es(x, ALPHA_ES), raw=False
)

risk_total = pd.DataFrame(
    {
        "date": var_total_series.index,
        "desk": "TOTAL",
        "var_99": var_total_series.values,
        "es_975": es_total_series.values,
    }
).dropna(subset=["var_99", "es_975"])

# Combine
risk_all = pd.concat([risk_desk, risk_total], ignore_index=True)

# Diversification benefit (VaR + ES)
pivot_var = risk_all.pivot(index="date", columns="desk", values="var_99").dropna()
pivot_es  = risk_all.pivot(index="date", columns="desk", values="es_975").dropna()

desk_cols = [c for c in pivot_var.columns if c != "TOTAL"]

div = pd.DataFrame(index=pivot_var.index)
div["sum_desk_var"] = pivot_var[desk_cols].sum(axis=1)
div["total_var"] = pivot_var["TOTAL"]
div["diversification_benefit_var"] = div["sum_desk_var"] - div["total_var"]

div["sum_desk_es"] = pivot_es[desk_cols].sum(axis=1)
div["total_es"] = pivot_es["TOTAL"]
div["diversification_benefit_es"] = div["sum_desk_es"] - div["total_es"]

# Write outputs
os.makedirs(OUT_DIR, exist_ok=True)
risk_all.to_csv(os.path.join(OUT_DIR, "risk_metrics.csv"), index=False)
div.reset_index().rename(columns={"index": "date"}).to_csv(os.path.join(OUT_DIR, "diversification.csv"), index=False)

# Sanity output
latest_date = div.index.max()
print("Latest snapshot:", latest_date.date())
print(div.loc[latest_date, ["total_var", "diversification_benefit_var", "total_es", "diversification_benefit_es"]])

Latest snapshot: 2026-05-29
total_var                      0.003187
diversification_benefit_var    0.002855
total_es                       0.003423
diversification_benefit_es     0.002499
Name: 2026-05-29 00:00:00, dtype: float64


### SVaR using Brexit YearWindow

In [6]:
PNL_TOTAL_PATH = os.path.join("data", "pnl_total.csv")
OUT_DIR = "data"

ALPHA_VAR = 0.99
ALPHA_ES = 0.975  # FRTB-style ES level

# Stress window (preferably around 12 months)
STRESS_START = "2020-02-15"
STRESS_END   = "2021-03-31"


def historical_var(series: pd.Series, alpha: float) -> float:
    q = np.quantile(series, 1 - alpha)
    return -q


def historical_es(series: pd.Series, alpha: float) -> float:
    q = np.quantile(series, 1 - alpha)
    tail = series[series <= q]
    if len(tail) == 0:
        return np.nan
    return -tail.mean()


os.makedirs(OUT_DIR, exist_ok=True)

pnl_total = pd.read_csv(PNL_TOTAL_PATH, parse_dates=["date"]).set_index("date").sort_index()

stress_sample = pnl_total.loc[STRESS_START:STRESS_END, "pnl_total"].dropna()

if stress_sample.empty:
    raise ValueError(f"No observations found in stress window {STRESS_START} to {STRESS_END}.")

if len(stress_sample) < 60:
    print(f"WARNING: Stress sample is small (n={len(stress_sample)}). SVaR/SES may be unstable.")

svar = historical_var(stress_sample, ALPHA_VAR)
ses  = historical_es(stress_sample, ALPHA_ES)

out = pd.DataFrame([{
    "scope": "TOTAL",
    "svar_99": float(svar),
    "ses_97_5": float(ses),
    "stress_start": STRESS_START,
    "stress_end": STRESS_END,
    "n_obs": int(len(stress_sample)),
}])

out_path = os.path.join(OUT_DIR, "stressed_metrics.csv")
out.to_csv(out_path, index=False)

print(f"Wrote stressed metrics to {out_path}")
print(out.to_string(index=False))

Wrote stressed metrics to data/stressed_metrics.csv
scope  svar_99  ses_97_5 stress_start stress_end  n_obs
TOTAL 0.007599  0.009105   2020-02-15 2021-03-31    273


### Backtesting, Exceptions and Kupiec Test

In [7]:
PNL_TOTAL_PATH = os.path.join("data", "pnl_total.csv")
RISK_METRICS_PATH = os.path.join("data", "risk_metrics.csv")
OUT_DIR = "data"

ALPHA_VAR = 0.99
BT_WINDOW = 250  # rolling window for exception stats


def kupiec_pof_test(n: int, x: int, alpha: float) -> tuple[float, float]:
    """
    Kupiec Proportion of Failures (POF) test.
    H0: exception rate = (1 - alpha)
    Returns: (LR_pof, p_value) with df=1 chi-square approx via erfc.
    """
    if n <= 0:
        return (float("nan"), float("nan"))

    eps = 1e-12
    p = 1.0 - alpha
    p = min(max(p, eps), 1.0 - eps)

    phat = x / n
    phat = min(max(phat, eps), 1.0 - eps)

    # LR formula: LR = 2 * [ (n-x)ln((1-phat)/(1-p)) + x ln(phat/p) ]
    lr = 0.0
    if n - x > 0:
        lr += 2.0 * (n - x) * (math.log(1.0 - phat) - math.log(1.0 - p))
    if x > 0:
        lr += 2.0 * x * (math.log(phat) - math.log(p))

    p_value = math.erfc(math.sqrt(max(lr, 0.0) / 2.0))
    return (float(lr), float(p_value))


os.makedirs(OUT_DIR, exist_ok=True)

# Load total P&L
pnl = pd.read_csv(PNL_TOTAL_PATH, parse_dates=["date"]).sort_values("date")
pnl = pnl.rename(columns={"pnl_total": "pnl"}).set_index("date")

# Load total VaR series from consolidated risk metrics output
risk_all = pd.read_csv(RISK_METRICS_PATH, parse_dates=["date"]).sort_values("date")
risk_total = risk_all[risk_all["desk"] == "TOTAL"].copy()
risk_total = risk_total.rename(columns={"var_99": "var"}).set_index("date")

# Align on overlapping dates (VaR starts after estimation window)
df = pnl.join(risk_total[["var"]], how="inner")
if df.empty:
    raise ValueError("No overlapping dates between pnl_total.csv and TOTAL VaR series in risk_metrics.csv")

# Exception: realized loss exceeds VaR (VaR is positive loss number)
df["exception"] = (df["pnl"] < -df["var"]).astype(int)

# Rolling exception stats
df["exceptions_rolling"] = df["exception"].rolling(BT_WINDOW, min_periods=BT_WINDOW).sum()
df["exception_rate_rolling"] = df["exceptions_rolling"] / BT_WINDOW

# Rolling Kupiec test p-values
lr_list = [np.nan] * len(df)
pv_list = [np.nan] * len(df)
exc_roll = df["exceptions_rolling"].to_numpy()

for i in range(len(df)):
    x = exc_roll[i]
    if not np.isfinite(x):
        continue
    lr, pv = kupiec_pof_test(BT_WINDOW, int(x), ALPHA_VAR)
    lr_list[i] = lr
    pv_list[i] = pv

df["kupiec_lr"] = lr_list
df["kupiec_pvalue"] = pv_list

out_path = os.path.join(OUT_DIR, "backtest_total.csv")
df.reset_index().to_csv(out_path, index=False)
print(f"Wrote {out_path} (rows={len(df)})")

# Latest snapshot
latest = df.iloc[-1]
expected = (1 - ALPHA_VAR) * BT_WINDOW

print("\nLatest backtest snapshot (TOTAL):")
print(f"Date: {df.index[-1].date()}")
print(f"VaR(99%): {latest['var']:.6f}")
print(f"P&L:      {latest['pnl']:.6f}")
print(f"Exception today: {int(latest['exception'])}")

if np.isfinite(latest["exceptions_rolling"]):
    print(f"Exceptions last {BT_WINDOW}d: {int(latest['exceptions_rolling'])} (expected ~{expected:.1f})")
    print(f"Rolling exception rate: {latest['exception_rate_rolling']:.3%}")
    print(f"Kupiec p-value: {latest['kupiec_pvalue']:.3f}")

total_exc = int(df["exception"].sum())
total_n = int(df["exception"].count())
print(f"\nOverall exceptions: {total_exc} over {total_n} days (rate {total_exc/total_n:.3%})")

Wrote data/backtest_total.csv (rows=3713)

Latest backtest snapshot (TOTAL):
Date: 2026-05-29
VaR(99%): 0.003187
P&L:      0.001520
Exception today: 0
Exceptions last 250d: 0 (expected ~2.5)
Rolling exception rate: 0.000%
Kupiec p-value: 0.025

Overall exceptions: 42 over 3713 days (rate 1.131%)


### Spikes and Outliers

In [8]:
PNL_DESK_PATH = os.path.join("data", "pnl_desk.csv")
PNL_TOTAL_PATH = os.path.join("data", "pnl_total.csv")
RISK_METRICS_PATH = os.path.join("data", "risk_metrics.csv")
OUT_DIR = "data"

# Tail detection (simple & explainable)
PNL_TAIL_Q = 0.01
ROLL_WINDOW = 250  # recent regime

# whether an outlier day lies in your stress calibration window
STRESS_START = "2020-02-15"
STRESS_END   = "2021-03-31"

os.makedirs(OUT_DIR, exist_ok=True)

# Load data
pnl_total = (
    pd.read_csv(PNL_TOTAL_PATH, parse_dates=["date"])
    .sort_values("date")
    .set_index("date")
)
pnl_desk = pd.read_csv(PNL_DESK_PATH, parse_dates=["date"])

risk_all = (
    pd.read_csv(RISK_METRICS_PATH, parse_dates=["date"])
    .sort_values("date")
)
var_total = (
    risk_all[risk_all["desk"] == "TOTAL"]
    .set_index("date")["var_99"]
    .sort_index()
)

pnl = pnl_total["pnl_total"].copy().sort_index()

# Rolling regime tails
low_roll = pnl.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).quantile(PNL_TAIL_Q)
high_roll = pnl.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).quantile(1 - PNL_TAIL_Q)
pnl_outlier_recent = (pnl <= low_roll) | (pnl >= high_roll)
pnl_outlier_recent = pnl_outlier_recent.fillna(False)

flag_dates = sorted(pnl_outlier_recent[pnl_outlier_recent].index)

rows = []
for d in flag_dates:
    row = {
        "date": d,
        #"pnl_outlier_recent": True,
        "pnl_total": float(pnl.loc[d]),
        "var_99": float(var_total.loc[d]) if d in var_total.index else np.nan,
        "in_stress_window": (pd.Timestamp(STRESS_START) <= d <= pd.Timestamp(STRESS_END)),
    }

    # top 3 desks by absolute P&L contribution
    g = pnl_desk[pnl_desk["date"] == d]
    if not g.empty:
        g = g.copy()
        g["abs_pnl"] = g["pnl"].abs()
        top = g.sort_values("abs_pnl", ascending=False).head(3)
        for i, (_, r) in enumerate(top.iterrows(), start=1):
            row[f"desk_{i}"] = r["desk"]
            row[f"desk_{i}_pnl"] = float(r["pnl"])

    rows.append(row)

flags_df = pd.DataFrame(rows).sort_values("date")

out_path = os.path.join(OUT_DIR, "validation_flags.csv")
flags_df.to_csv(out_path, index=False)
print(f"Wrote {out_path} (rows={len(flags_df)})")

print("\nLatest flagged tail events:")
print(flags_df.tail(10).to_string(index=False))

Wrote data/validation_flags.csv (rows=96)

Latest flagged tail events:
      date  pnl_total   var_99  in_stress_window desk_1  desk_1_pnl desk_2  desk_2_pnl desk_3  desk_3_pnl
2025-02-21   0.004793 0.004058             False     FX    0.003107 Credit    0.001433  Rates    0.000254
2025-03-06   0.004680 0.004058             False     FX    0.004552  Rates    0.000311 Credit   -0.000184
2025-03-10   0.004684 0.004058             False     FX    0.002021 Credit    0.001780  Rates    0.000883
2025-04-03   0.009254 0.004058             False     FX    0.003886  Rates    0.002910 Credit    0.002458
2025-04-04   0.006073 0.004058             False     FX    0.004205 Credit    0.002520  Rates   -0.000652
2025-04-07  -0.004566 0.004371             False Credit   -0.004127     FX   -0.000794  Rates    0.000355
2025-04-08  -0.005001 0.004616             False     FX   -0.003195 Credit   -0.002877  Rates    0.001071
2025-04-11   0.006011 0.004616             False     FX    0.009457  Rates   -0.0

### Scenario Stress Engine

In [9]:
## Config ##
POSITIONS_PATH = "positions_stress_ready.csv"
RETURNS_PATH = os.path.join("data", "returns.csv")
TICKER_META_PATH = os.path.join("data", "ticker_metadata.csv")
ASOF_META_PATH = os.path.join("data", "asof_metadata.csv")
OUT_DIR = "data"

# historical window used to auto-build a "worst day" historical scenario
HIST_STRESS_START = "2020-02-15"
HIST_STRESS_END   = "2021-03-31"

os.makedirs(OUT_DIR, exist_ok=True)

def load_positions(path: str) -> pd.DataFrame:
    pos = pd.read_csv(path)

    required = {"desk", "ticker", "weight", "risk_type", "risk_factor", "bucket"}
    missing = required - set(pos.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {sorted(missing)}")

    # Aggregate potential duplicates safely
    pos = (
        pos.groupby(["desk", "ticker", "risk_type", "risk_factor", "bucket"], as_index=False)["weight"]
        .sum()
        .sort_values(["desk", "ticker"])
    )
    return pos


def load_returns(path: str, tickers: list[str]) -> pd.DataFrame:
    rets = pd.read_csv(path, parse_dates=["date"]).set_index("date").sort_index()

    missing = [t for t in tickers if t not in rets.columns]
    if missing:
        raise ValueError(f"returns.csv missing tickers: {missing}")

    # Enforce complete dates
    rets = rets[tickers]
    complete = rets.notna().all(axis=1)
    rets = rets.loc[complete]
    if rets.empty:
        raise ValueError("No complete dates in returns after filtering to required tickers.")

    return rets


def pick_asof_date(returns: pd.DataFrame) -> pd.Timestamp:
    # Prefer safe_asof_date from asof_metadata if available, else latest complete date in returns
    if os.path.exists(ASOF_META_PATH):
        meta = pd.read_csv(ASOF_META_PATH)
        if not meta.empty:
            safe = meta.loc[0].get("safe_asof_date", "")
            true = meta.loc[0].get("true_asof_date", "")
            for d in [safe, true]:
                if isinstance(d, str) and d.strip():
                    ts = pd.Timestamp(d)
                    if ts in returns.index:
                        return ts
    return returns.index.max()


def scenario_defs_default() -> dict[str, dict]:
    """
    Scenarios defined in return space, keyed by risk_factor (preferred) or ticker.
    Shocks are single-period returns, e.g. -0.15 = -15%.

    You can tune these magnitudes — the point is consistent structure + attribution.
    """
    return {
        # Rates shocks applied to USD_curve bucketed proxies (IEF/TLT)
        "Rates_Parallel_200bp": {
            "type": "risk_factor",
            "shocks": {"USD_curve": {"default": -0.06, "by_bucket": {"USD_5Y": -0.04, "USD_20Y": -0.15}}},
            "notes": "Simple proxy: rates sell-off; larger impact at long end.",
        },
        "Rates_Steepener": {
            "type": "risk_factor",
            "shocks": {"USD_curve": {"default": -0.04, "by_bucket": {"USD_5Y": -0.02, "USD_20Y": -0.12}}},
            "notes": "Steepening shock: long end worse than belly.",
        },

        # Credit spread widening
        "Credit_IG_+150bp": {
            "type": "risk_factor",
            "shocks": {"IG_spread": {"default": -0.06}},
            "notes": "IG spread widening proxy via LQD return shock.",
        },
        "Credit_HY_+400bp": {
            "type": "risk_factor",
            "shocks": {"HY_spread": {"default": -0.12}},
            "notes": "HY spread widening proxy via HYG return shock.",
        },

        # FX shocks
        "FX_EURUSD_-15pct": {
            "type": "risk_factor",
            "shocks": {"EURUSD": {"default": -0.15}},
            "notes": "EUR depreciates vs USD.",
        },
        "FX_USDJPY_+10pct": {
            "type": "risk_factor",
            "shocks": {"USDJPY": {"default": +0.10}},
            "notes": "USD strengthens vs JPY (USDJPY up).",
        },

        # Combined macro scenario (rates + credit + FX)
        "Macro_Combined": {
            "type": "mixed",
            "shocks": {
                "risk_factor": {
                    "USD_curve": {"default": -0.08, "by_bucket": {"USD_5Y": -0.05, "USD_20Y": -0.18}},
                    "IG_spread": {"default": -0.07},
                    "HY_spread": {"default": -0.15},
                    "EURUSD": {"default": -0.12},
                    "USDJPY": {"default": +0.07},
                }
            },
            "notes": "Severe macro: rates sell-off, spread widening, FX moves.",
        },
    }


def build_historical_worst_day_scenario(
    name: str,
    returns: pd.DataFrame,
    positions: pd.DataFrame,
    start: str,
    end: str,
) -> tuple[str, dict] | None:
    """
    Construct a historical scenario using the *worst* realized total P&L day in a window,
    with per-ticker returns on that day used as the shocks.
    """
    window = returns.loc[start:end].copy()
    if window.empty:
        return None

    w = positions.groupby("ticker")["weight"].sum()
    pnl_total = window.mul(w, axis=1).sum(axis=1)
    worst_date = pnl_total.idxmin()
    shocks = window.loc[worst_date].to_dict()

    scen = {
        "type": "ticker",
        "shocks": shocks,  
        "notes": f"Worst realized total P&L day between {start} and {end}: {worst_date.date()}",
        "historical_date": str(worst_date.date()),
    }
    return name, scen


def apply_scenario(
    positions: pd.DataFrame,
    scenario: dict,
) -> pd.DataFrame:
    """
    Returns instrument-level results with columns:
      scenario, ticker, desk, risk_type, risk_factor, bucket, weight, shock_return, pnl
    """
    pos = positions.copy()

    scen_type = scenario.get("type", "ticker")
    shocks = scenario.get("shocks", {})

    shock_return = pd.Series(0.0, index=pos.index)

    if scen_type == "ticker":
        shock_map = shocks
        shock_return = pos["ticker"].map(shock_map).fillna(0.0)

    elif scen_type == "risk_factor":
        rf_shocks = shocks
        for i, r in pos.iterrows():
            rf = r["risk_factor"]
            bucket = r["bucket"]
            if rf in rf_shocks:
                d = rf_shocks[rf]
                if isinstance(d, dict):
                    by_bucket = d.get("by_bucket", {})
                    shock_return.iloc[i] = float(by_bucket.get(bucket, d.get("default", 0.0)))
                else:
                    shock_return.iloc[i] = float(d)
            else:
                shock_return.iloc[i] = 0.0

    elif scen_type == "mixed":
        rf_shocks = shocks.get("risk_factor", {})
        t_shocks = shocks.get("ticker", {})

        for i, r in pos.iterrows():
            t = r["ticker"]
            rf = r["risk_factor"]
            bucket = r["bucket"]

            if t in t_shocks:
                shock_return.iloc[i] = float(t_shocks[t])
                continue

            if rf in rf_shocks:
                d = rf_shocks[rf]
                if isinstance(d, dict):
                    by_bucket = d.get("by_bucket", {})
                    shock_return.iloc[i] = float(by_bucket.get(bucket, d.get("default", 0.0)))
                else:
                    shock_return.iloc[i] = float(d)
            else:
                shock_return.iloc[i] = 0.0
    else:
        raise ValueError(f"Unknown scenario type: {scen_type}")

    pos["shock_return"] = shock_return.astype(float)
    pos["pnl"] = pos["weight"].astype(float) * pos["shock_return"].astype(float)
    return pos


positions = load_positions(POSITIONS_PATH)
tickers = positions["ticker"].unique().tolist()
returns = load_returns(RETURNS_PATH, tickers)
asof_date = pick_asof_date(returns)

# Build scenario set
scenarios = scenario_defs_default()

# Add a historical worst-day scenario from a chosen stress window
hist = build_historical_worst_day_scenario(
    name="Historical_WorstDay_Window",
    returns=returns,
    positions=positions,
    start=HIST_STRESS_START,
    end=HIST_STRESS_END,
)
if hist is not None:
    k, v = hist
    scenarios[k] = v

# Evaluate scenarios
instrument_rows = []
scenario_meta_rows = []

for scen_name, scen in scenarios.items():
    inst = apply_scenario(positions, scen)
    inst.insert(0, "scenario", scen_name)
    instrument_rows.append(inst)

    scenario_meta_rows.append(
        {
            "scenario": scen_name,
            "type": scen.get("type", ""),
            "notes": scen.get("notes", ""),
            "historical_date": scen.get("historical_date", ""),
            "asof_date_used": str(asof_date.date()),
        }
    )

scenario_instrument = pd.concat(instrument_rows, ignore_index=True)

# Aggregations
scenario_desk = (
    scenario_instrument.groupby(["scenario", "desk"], as_index=False)["pnl"]
    .sum()
    .sort_values(["scenario", "pnl"])
)

scenario_factor = (
    scenario_instrument.groupby(["scenario", "risk_factor"], as_index=False)["pnl"]
    .sum()
    .sort_values(["scenario", "pnl"])
)

scenario_total = (
    scenario_instrument.groupby(["scenario"], as_index=False)["pnl"]
    .sum()
    .rename(columns={"pnl": "pnl_total"})
    .sort_values("pnl_total")
)

scenario_defs = pd.DataFrame(scenario_meta_rows).sort_values("scenario")

# outputs
scenario_instrument.to_csv(os.path.join(OUT_DIR, "scenario_results_instrument.csv"), index=False)
scenario_desk.to_csv(os.path.join(OUT_DIR, "scenario_results_desk.csv"), index=False)
scenario_factor.to_csv(os.path.join(OUT_DIR, "scenario_results_factor.csv"), index=False)
scenario_total.to_csv(os.path.join(OUT_DIR, "scenario_results_total.csv"), index=False)
scenario_defs.to_csv(os.path.join(OUT_DIR, "scenario_definitions.csv"), index=False)

print("Wrote scenario stress outputs:")
print(" - data/scenario_results_instrument.csv")
print(" - data/scenario_results_desk.csv")
print(" - data/scenario_results_factor.csv")
print(" - data/scenario_results_total.csv")
print(" - data/scenario_definitions.csv")

print("\nScenario totals (sorted worst to best):")
print(scenario_total.to_string(index=False))

print("\nWorst scenario by total loss:")
worst = scenario_total.iloc[0]
print(worst.to_string(index=False))

Wrote scenario stress outputs:
 - data/scenario_results_instrument.csv
 - data/scenario_results_desk.csv
 - data/scenario_results_factor.csv
 - data/scenario_results_total.csv
 - data/scenario_definitions.csv

Scenario totals (sorted worst to best):
                  scenario  pnl_total
          FX_EURUSD_-15pct  -0.037500
            Macro_Combined  -0.018500
          Credit_IG_+150bp  -0.018000
Historical_WorstDay_Window  -0.015404
          FX_USDJPY_+10pct  -0.010000
      Rates_Parallel_200bp   0.008500
           Rates_Steepener   0.011000
          Credit_HY_+400bp   0.024000

Worst scenario by total loss:
FX_EURUSD_-15pct
         -0.0375


### Stress Attribution + Concentration

In [10]:
OUT_DIR = "data"

INSTR_PATH = os.path.join(OUT_DIR, "scenario_results_instrument.csv")
DESK_PATH  = os.path.join(OUT_DIR, "scenario_results_desk.csv")
FACT_PATH  = os.path.join(OUT_DIR, "scenario_results_factor.csv")
TOTAL_PATH = os.path.join(OUT_DIR, "scenario_results_total.csv")
DEF_PATH   = os.path.join(OUT_DIR, "scenario_definitions.csv")

os.makedirs(OUT_DIR, exist_ok=True)

## Load ##
inst = pd.read_csv(INSTR_PATH)
desk = pd.read_csv(DESK_PATH)
fact = pd.read_csv(FACT_PATH)
tot  = pd.read_csv(TOTAL_PATH)
defs = pd.read_csv(DEF_PATH) if os.path.exists(DEF_PATH) else pd.DataFrame()

# Ensure types
for df in [inst, desk, fact, tot]:
    if "pnl" in df.columns:
        df["pnl"] = df["pnl"].astype(float)
    if "pnl_total" in df.columns:
        df["pnl_total"] = df["pnl_total"].astype(float)

def top_n_contributors(df: pd.DataFrame, group_cols: list[str], value_col: str, n: int, label: str) -> pd.DataFrame:
    """
    For each scenario, select top N contributors by absolute value of value_col.
    Returns long table with contributor name and signed pnl.
    """
    out_rows = []
    for scen, g in df.groupby("scenario"):
        g = g.copy()
        g["_abs"] = g[value_col].abs()
        g = g.sort_values("_abs", ascending=False).head(n)
        
        for rank, (_, r) in enumerate(g.iterrows(), start=1):
            name = " | ".join(str(r[c]) for c in group_cols)
            out_rows.append(
                {
                    "scenario": scen,
                    "contributor_type": label,
                    "rank": rank,
                    "contributor": name,
                    "pnl": float(r[value_col]),
                    "abs_pnl": float(abs(r[value_col])),
                }
            )
    return pd.DataFrame(out_rows)
    
def top_n_contributors_alt(df: pd.DataFrame, group_cols: list[str], value_col: str, n: int, label: str) -> pd.DataFrame:
    """
    For each scenario, select top N contributors by absolute value of value_col.
    Returns long table with contributor name and signed pnl.
    """
    out_rows = []
    for scen, g in df.groupby("scenario"):
        g = g.copy()
        g["_abs"] = g[value_col].abs()
        #g = g.sort_values("_abs", ascending=False).head(n)
        g = g.sort_values(value_col, ascending=True).head(n)
        
        for rank, (_, r) in enumerate(g.iterrows(), start=1):
            name = " | ".join(str(r[c]) for c in group_cols)
            out_rows.append(
                {
                    "scenario": scen,
                    "contributor_type": label,
                    "rank": rank,
                    "contributor": name,
                    "pnl": float(r[value_col]),
                    "abs_pnl": float(abs(r[value_col])),
                }
            )
    return pd.DataFrame(out_rows)


def concentration_metrics(df: pd.DataFrame, key_col: str, value_col: str, prefix: str) -> pd.DataFrame:
    """
    For each scenario compute:
      - top1_share_abs, top3_share_abs, top5_share_abs: sum(abs pnl of top k) / sum(abs pnl)
      - hhi_abs: sum(share^2) on abs shares (0..1). Higher = more concentrated.
      - n_contributors
    """
    rows = []
    for scen, g in df.groupby("scenario"):
        vals = g[value_col].astype(float).values
        abs_vals = np.abs(vals)
        denom = abs_vals.sum()
        if denom == 0:
            rows.append(
                {"scenario": scen, f"{prefix}_n": int(len(g)),
                 f"{prefix}_top1_share_abs": np.nan, f"{prefix}_top3_share_abs": np.nan, f"{prefix}_top5_share_abs": np.nan,
                 f"{prefix}_hhi_abs": np.nan}
            )
            continue

        # Sort by abs contribution
        abs_sorted = np.sort(abs_vals)[::-1]
        shares = abs_vals / denom
        hhi = float(np.sum(shares**2))

        def topk(k: int) -> float:
            return float(abs_sorted[:k].sum() / denom) if len(abs_sorted) >= 1 else np.nan

        rows.append(
            {
                "scenario": scen,
                f"{prefix}_n": int(len(g)),
                f"{prefix}_top1_share_abs": topk(1),
                f"{prefix}_top3_share_abs": topk(3),
                f"{prefix}_top5_share_abs": topk(5),
                f"{prefix}_hhi_abs": hhi,
            }
        )
    return pd.DataFrame(rows)


def make_waterfall(df: pd.DataFrame, group_cols: list[str], value_col: str, label: str, top_k: int = 10) -> pd.DataFrame:
    """
    For plotting: keep top_k contributors by abs pnl per scenario, aggregate the rest as 'OTHER'.
    """
    rows = []
    for scen, g in df.groupby("scenario"):
        g = g.copy()
        g["name"] = g.apply(lambda r: " | ".join(str(r[c]) for c in group_cols), axis=1)
        g["_abs"] = g[value_col].abs()
        g = g.sort_values("_abs", ascending=False)

        top = g.head(top_k)
        rest = g.iloc[top_k:]

        for _, r in top.iterrows():
            rows.append(
                {
                    "scenario": scen,
                    "component_type": label,
                    "component": r["name"],
                    "pnl": float(r[value_col]),
                    "abs_pnl": float(abs(r[value_col])),
                    "is_other": False,
                }
            )

        if len(rest) > 0:
            rows.append(
                {
                    "scenario": scen,
                    "component_type": label,
                    "component": "OTHER",
                    "pnl": float(rest[value_col].sum()),
                    "abs_pnl": float(rest[value_col].abs().sum()),
                    "is_other": True,
                }
            )

    return pd.DataFrame(rows)


## Top contributors tables ##
top_instruments = top_n_contributors(
    inst,
    group_cols=["desk", "ticker"],
    value_col="pnl",
    n=5,
    label="INSTRUMENT",
)

top_desks = top_n_contributors(
    desk,
    group_cols=["desk"],
    value_col="pnl",
    n=3,
    label="DESK",
)

top_factors = top_n_contributors(
    fact,
    group_cols=["risk_factor"],
    value_col="pnl",
    n=5,
    label="RISK_FACTOR",
)

top_all = pd.concat([top_desks, top_factors, top_instruments], ignore_index=True)

# total pnl
tot_map = tot.set_index("scenario")["pnl_total"].to_dict()
top_all["scenario_total_pnl"] = top_all["scenario"].map(tot_map).astype(float)

top_all = top_all.sort_values(["scenario", "contributor_type", "rank"])

## Concentration metrics ##
cm_desk = concentration_metrics(desk, key_col="desk", value_col="pnl", prefix="desk")
cm_fact = concentration_metrics(fact, key_col="risk_factor", value_col="pnl", prefix="factor")
cm_inst = concentration_metrics(inst, key_col="ticker", value_col="pnl", prefix="instrument")

conc = cm_desk.merge(cm_fact, on="scenario", how="outer").merge(cm_inst, on="scenario", how="outer")
conc = conc.merge(tot, on="scenario", how="left")
if not defs.empty:
    conc = conc.merge(defs, on="scenario", how="left")

# worst scenario ranking
conc = conc.sort_values("pnl_total").reset_index(drop=True)
conc["worst_rank_by_total_loss"] = conc.index + 1

## Waterfall components for HTML charts ##
wf_desk = make_waterfall(desk, group_cols=["desk"], value_col="pnl", label="DESK", top_k=10)
wf_fact = make_waterfall(fact, group_cols=["risk_factor"], value_col="pnl", label="RISK_FACTOR", top_k=10)
wf_inst = make_waterfall(inst, group_cols=["desk", "ticker"], value_col="pnl", label="INSTRUMENT", top_k=10)
waterfall = pd.concat([wf_desk, wf_fact, wf_inst], ignore_index=True)

## Write outputs ##
top_path = os.path.join(OUT_DIR, "scenario_top_contributors.csv")
conc_path = os.path.join(OUT_DIR, "scenario_concentration_metrics.csv")
wf_path = os.path.join(OUT_DIR, "scenario_waterfall_components.csv")

top_all.to_csv(top_path, index=False)
conc.to_csv(conc_path, index=False)
waterfall.to_csv(wf_path, index=False)

print("Wrote stress attribution outputs:")
print(f" - {top_path} (rows={len(top_all)})")
print(f" - {conc_path} (rows={len(conc)})")
print(f" - {wf_path} (rows={len(waterfall)})")

print("\nWorst 5 scenarios by total loss:")
print(conc[["scenario", "pnl_total", "desk_top3_share_abs", "factor_top3_share_abs", "instrument_top3_share_abs"]].head(5).to_string(index=False))

print("\nSample top contributors (first scenario alphabetically):")
first_scen = sorted(top_all["scenario"].unique().tolist())[0] if len(top_all) else None
if first_scen:
    print(top_all[top_all["scenario"] == first_scen].head(15).to_string(index=False))

Wrote stress attribution outputs:
 - data/scenario_top_contributors.csv (rows=104)
 - data/scenario_concentration_metrics.csv (rows=8)
 - data/scenario_waterfall_components.csv (rows=112)

Worst 5 scenarios by total loss:
                  scenario  pnl_total  desk_top3_share_abs  factor_top3_share_abs  instrument_top3_share_abs
          FX_EURUSD_-15pct  -0.037500                  1.0               1.000000                   1.000000
            Macro_Combined  -0.018500                  1.0               0.830769                   0.656604
          Credit_IG_+150bp  -0.018000                  1.0               1.000000                   1.000000
Historical_WorstDay_Window  -0.015404                  1.0               0.912248                   0.876772
          FX_USDJPY_+10pct  -0.010000                  1.0               1.000000                   1.000000

Sample top contributors (first scenario alphabetically):
        scenario contributor_type  rank  contributor    pnl  abs_p

### Portfolio Change Attribution

In [11]:
OUT_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)

## Inputs ##
BASE_POSITIONS_PATH = "positions_stress_ready.csv"
P0_PATH = "positions_T0.csv"
P1_PATH = "positions_T1.csv"

SCEN_INST_PATH = os.path.join(OUT_DIR, "scenario_results_instrument.csv")
if not os.path.exists(SCEN_INST_PATH):
    raise FileNotFoundError(
        f"Missing {SCEN_INST_PATH}. Run the scenario stress engine first so we have scenario-by-instrument shocks."
    )

REQUIRED_COLS = {"desk", "ticker", "weight", "risk_type", "risk_factor", "bucket"}

# Governance-style caps
MAX_GROSS = 2.5
MAX_DESK_GROSS = 1.5

def load_positions(path: str) -> pd.DataFrame:
    pos = pd.read_csv(path)
    missing = REQUIRED_COLS - set(pos.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {sorted(missing)}")
    pos = (
        pos.groupby(["desk", "ticker", "risk_type", "risk_factor", "bucket"], as_index=False)["weight"]
        .sum()
        .sort_values(["desk", "ticker"])
    )
    return pos


def governance_ok(pos: pd.DataFrame) -> tuple[bool, str]:
    gross = float(pos["weight"].abs().sum())
    if gross > MAX_GROSS:
        return False, f"Gross exposure {gross:.3f} exceeds MAX_GROSS={MAX_GROSS}"

    desk_gross = pos.groupby("desk")["weight"].apply(lambda x: float(x.abs().sum()))
    if (desk_gross > MAX_DESK_GROSS).any():
        bad = desk_gross[desk_gross > MAX_DESK_GROSS]
        return False, f"Desk gross exposure exceeds cap {MAX_DESK_GROSS}: {bad.to_dict()}"

    return True, "OK"


def create_T0_T1_if_missing() -> None:
    if not os.path.exists(BASE_POSITIONS_PATH):
        raise FileNotFoundError(f"Missing {BASE_POSITIONS_PATH}. Create it by running your positions step first.")

    base = load_positions(BASE_POSITIONS_PATH)

    # Create T0 as exact copy if missing
    if not os.path.exists(P0_PATH):
        base.to_csv(P0_PATH, index=False)
        print(f"Created {P0_PATH} from {BASE_POSITIONS_PATH}")

    # Create T1 with realistic tweaks if missing
    if not os.path.exists(P1_PATH):
        p1 = base.copy()

        def bump(ticker: str, delta: float):
            m = p1["ticker"] == ticker
            if not m.any():
                raise ValueError(f"Ticker {ticker} not found in base positions for T1 creation.")
            p1.loc[m, "weight"] = p1.loc[m, "weight"] + delta

        bump("IEF", +0.05)      # more intermediate rates
        bump("HYG", +0.05)      # less short HY (reduce hedge)
        bump("EURUSD", +0.03)   # more EURUSD

        ok, msg = governance_ok(p1)
        if not ok:
            # If caps break, scale all weights down slightly to fit gross cap
            scale = MAX_GROSS / float(p1["weight"].abs().sum())
            p1["weight"] = p1["weight"] * scale
            ok2, msg2 = governance_ok(p1)
            if not ok2:
                raise ValueError(f"Auto-created T1 violates governance even after scaling: {msg2}")

        p1.to_csv(P1_PATH, index=False)
        print(f"Created {P1_PATH} with a small set of position changes")


def build_universe(p0: pd.DataFrame, p1: pd.DataFrame) -> pd.DataFrame:
    base = pd.concat([p0, p1], ignore_index=True)
    base = base.sort_values(["ticker"]).drop_duplicates(subset=["ticker"], keep="last")
    return base[["ticker", "desk", "risk_type", "risk_factor", "bucket"]].copy().sort_values(["desk", "ticker"])


def align_weights(universe: pd.DataFrame, p: pd.DataFrame, label: str) -> pd.Series:
    w = p.groupby("ticker")["weight"].sum()
    out = universe["ticker"].map(w).fillna(0.0).astype(float)
    out.index = universe["ticker"]
    out.name = f"weight_{label}"
    return out


create_T0_T1_if_missing()

p0 = load_positions(P0_PATH)
p1 = load_positions(P1_PATH)

universe = build_universe(p0, p1)

w0 = align_weights(universe, p0, "T0")
w1 = align_weights(universe, p1, "T1")
dw = (w1 - w0).rename("delta_weight")

# Load scenario-by-instrument shocks from scenario engine run
scen_inst = pd.read_csv(SCEN_INST_PATH)

needed = {"scenario", "ticker", "shock_return"}
missing = needed - set(scen_inst.columns)
if missing:
    raise ValueError(f"{SCEN_INST_PATH} missing columns: {sorted(missing)}")

scen_inst = scen_inst[scen_inst["ticker"].isin(universe["ticker"])].copy()
scen_inst = scen_inst[["scenario", "ticker", "shock_return"]].drop_duplicates()
scen_inst["shock_return"] = scen_inst["shock_return"].astype(float)

# Merge metadata for attribution
meta = universe.set_index("ticker")
scen_inst = scen_inst.merge(meta.reset_index(), on="ticker", how="left")

# Compute stress under T0, T1 and delta (position effect)
scen_inst["weight_T0"] = scen_inst["ticker"].map(w0).astype(float)
scen_inst["weight_T1"] = scen_inst["ticker"].map(w1).astype(float)
scen_inst["delta_weight"] = scen_inst["ticker"].map(dw).astype(float)

scen_inst["pnl_T0"] = scen_inst["weight_T0"] * scen_inst["shock_return"]
scen_inst["pnl_T1"] = scen_inst["weight_T1"] * scen_inst["shock_return"]
scen_inst["delta_pnl"] = scen_inst["delta_weight"] * scen_inst["shock_return"]

# Total
"""
tot = (
    scen_inst.groupby("scenario", as_index=False)[["pnl_T0", "pnl_T1", "delta_pnl"]]
    .sum()
    .rename(columns={"pnl_T0": "stress_T0", "pnl_T1": "stress_T1", "delta_pnl": "delta_stress"})
)
tot["delta_check"] = tot["stress_T1"] - tot["stress_T0"]
tot["delta_gap"] = tot["delta_stress"] - tot["delta_check"]
tot = tot.sort_values("delta_stress")
"""
tot = (
    scen_inst.groupby("scenario", as_index=False)[["pnl_T0", "pnl_T1"]]
    .sum()
    .rename(columns={"pnl_T0": "stress_T0", "pnl_T1": "stress_T1"})
)

# Change
tot["delta_stress"] = tot["stress_T1"] - tot["stress_T0"]

# Desk / factor / instrument attribution
desk_attr = (
    scen_inst.groupby(["scenario", "desk"], as_index=False)["delta_pnl"]
    .sum()
    .rename(columns={"delta_pnl": "delta_stress"})
    .sort_values(["scenario", "delta_stress"])
)

factor_attr = (
    scen_inst.groupby(["scenario", "risk_factor"], as_index=False)["delta_pnl"]
    .sum()
    .rename(columns={"delta_pnl": "delta_stress"})
    .sort_values(["scenario", "delta_stress"])
)

inst_attr = (
    scen_inst.groupby(["scenario", "ticker"], as_index=False)["delta_pnl"]
    .sum()
    .rename(columns={"delta_pnl": "delta_stress"})
    .sort_values(["scenario", "delta_stress"])
)

# Write outputs
tot.to_csv(os.path.join(OUT_DIR, "stress_change_attribution_total.csv"), index=False)
desk_attr.to_csv(os.path.join(OUT_DIR, "stress_change_attribution_desk.csv"), index=False)
factor_attr.to_csv(os.path.join(OUT_DIR, "stress_change_attribution_factor.csv"), index=False)
inst_attr.to_csv(os.path.join(OUT_DIR, "stress_change_attribution_instrument.csv"), index=False)

print("Wrote portfolio change attribution outputs:")
print(" - data/stress_change_attribution_total.csv")
print(" - data/stress_change_attribution_desk.csv")
print(" - data/stress_change_attribution_factor.csv")
print(" - data/stress_change_attribution_instrument.csv")

print("\nCreated/used portfolios:")
print(f" - {P0_PATH}")
print(f" - {P1_PATH}")

print("\nTop scenarios by worsening (most negative delta_stress):")
#print(tot.head(10)[["scenario", "stress_T0", "stress_T1", "delta_stress", "delta_gap"]].to_string(index=False))
print(tot.head(10)[["scenario", "stress_T0", "stress_T1", "delta_stress"]].to_string(index=False))

Wrote portfolio change attribution outputs:
 - data/stress_change_attribution_total.csv
 - data/stress_change_attribution_desk.csv
 - data/stress_change_attribution_factor.csv
 - data/stress_change_attribution_instrument.csv

Created/used portfolios:
 - positions_T0.csv
 - positions_T1.csv

Top scenarios by worsening (most negative delta_stress):
                  scenario  stress_T0  stress_T1  delta_stress
          Credit_HY_+400bp   0.024000   0.018000     -0.006000
          Credit_IG_+150bp  -0.018000  -0.018000      0.000000
          FX_EURUSD_-15pct  -0.037500  -0.042000     -0.004500
          FX_USDJPY_+10pct  -0.010000  -0.010000      0.000000
Historical_WorstDay_Window  -0.015404  -0.016511     -0.001106
            Macro_Combined  -0.018500  -0.032100     -0.013600
      Rates_Parallel_200bp   0.008500   0.006500     -0.002000
           Rates_Steepener   0.011000   0.010000     -0.001000


### RNIM Quantification

In [12]:
OUT_DIR = "data"

RISK_PATH = os.path.join(OUT_DIR, "risk_metrics.csv")
STRESS_PATH = os.path.join(OUT_DIR, "stressed_metrics.csv")
CONC_PATH = os.path.join(OUT_DIR, "scenario_concentration_metrics.csv")

os.makedirs(OUT_DIR, exist_ok=True)

## Load inputs ##
risk = pd.read_csv(RISK_PATH)
stress = pd.read_csv(STRESS_PATH)
conc = pd.read_csv(CONC_PATH)

# Latest TOTAL ES
risk_total = risk[risk["desk"] == "TOTAL"].sort_values("date")
latest_es = float(risk_total.iloc[-1]["es_975"])

# Stressed ES
stressed_es = float(stress.iloc[0]["ses_97_5"])

# Base capital
base_capital = 1.5 * max(latest_es, stressed_es)

## RNIM Add-ons ##

# Liquidity Risk Add-on
# Proxy: 10% of stressed ES
liquidity_addon = 0.10 * stressed_es

# Concentration Risk Add-on
# If instrument top1 share > 50%, add 5% of stressed ES
conc_latest = conc.sort_values("pnl_total").iloc[0]  # worst scenario
top1_share = float(conc_latest.get("instrument_top1_share_abs", 0.0))

if top1_share > 0.50:
    concentration_addon = 0.05 * stressed_es
else:
    concentration_addon = 0.0

# Jump-to-Default Proxy
# If HY contribution share > 40% in worst scenario → add 5%
hy_share = float(conc_latest.get("factor_top1_share_abs", 0.0))

if hy_share > 0.40:
    jtd_addon = 0.05 * stressed_es
else:
    jtd_addon = 0.0

# Correlation Breakdown Add-on
# If desk_top3_share_abs > 80% (stress highly concentrated across desks)
desk_top1 = float(conc_latest.get("desk_top1_share_abs", 0.0))

if desk_top1 > 0.80:
    corr_addon = 0.05 * stressed_es
else:
    corr_addon = 0.0

## Total Capital ##
rnim_total = liquidity_addon + concentration_addon + jtd_addon + corr_addon

capital_with_rnim = base_capital + rnim_total

out = pd.DataFrame([{
    "latest_es_975": latest_es,
    "stressed_es_975": stressed_es,
    "base_capital_model": base_capital,
    "liquidity_addon": liquidity_addon,
    "concentration_addon": concentration_addon,
    "jump_to_default_addon": jtd_addon,
    "correlation_breakdown_addon": corr_addon,
    "total_rnim_addon": rnim_total,
    "capital_with_rnim": capital_with_rnim,
}])

out_path = os.path.join(OUT_DIR, "capital_with_rnim.csv")
out.to_csv(out_path, index=False)

print("Wrote capital overlay with RNIM:")
print(out.to_string(index=False))

Wrote capital overlay with RNIM:
 latest_es_975  stressed_es_975  base_capital_model  liquidity_addon  concentration_addon  jump_to_default_addon  correlation_breakdown_addon  total_rnim_addon  capital_with_rnim
      0.003423         0.009105            0.013658         0.000911             0.000455               0.000455                     0.000455          0.002276           0.015934


### Reverse Stress Test

In [13]:
DATA_DIR = "data"
OUT_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)

SCEN_TOTAL_PATH = os.path.join(DATA_DIR, "scenario_results_total.csv")
SCEN_DEF_PATH = os.path.join(DATA_DIR, "scenario_definitions.csv")
CAP_RNIM_PATH = os.path.join(DATA_DIR, "capital_with_rnim.csv")

if not os.path.exists(SCEN_TOTAL_PATH):
    raise FileNotFoundError(f"Missing {SCEN_TOTAL_PATH}. Run the scenario stress engine first.")

scen_total = pd.read_csv(SCEN_TOTAL_PATH)
if "scenario" not in scen_total.columns or "pnl_total" not in scen_total.columns:
    raise ValueError("scenario_results_total.csv must contain columns: scenario, pnl_total")

scen_total["pnl_total"] = scen_total["pnl_total"].astype(float)

## Choose a target loss threshold ##

TARGET_LOSS = None  # set to a negative number to override

if TARGET_LOSS is None and os.path.exists(CAP_RNIM_PATH):
    cap = pd.read_csv(CAP_RNIM_PATH)
    if not cap.empty and "capital_with_rnim" in cap.columns:
        TARGET_LOSS = -float(cap.iloc[0]["capital_with_rnim"])

if TARGET_LOSS is None:
    # Fallback default if no capital file exists
    TARGET_LOSS = -0.02  

if TARGET_LOSS >= 0:
    raise ValueError("TARGET_LOSS must be negative (a loss threshold).")

## Compute required multipliers ##
df = scen_total.copy()
df["target_loss"] = float(TARGET_LOSS)

# Only meaningful for scenarios that produce a loss (negative pnl_total)
df["multiplier_required"] = np.where(
    df["pnl_total"] < 0,
    df["target_loss"] / df["pnl_total"],  # negative / negative => positive multiplier
    np.nan
)

# Interpretation helpers
df["already_breaches"] = df["multiplier_required"] <= 1.0
df["multiplier_required"] = df["multiplier_required"].astype(float)

# Add metadata if present
if os.path.exists(SCEN_DEF_PATH):
    defs = pd.read_csv(SCEN_DEF_PATH)
    if "scenario" in defs.columns:
        df = df.merge(defs, on="scenario", how="left")

# Rank by fragility: smallest multiplier (closest to breach)
df["fragility_rank"] = df["multiplier_required"].rank(method="dense", ascending=True)

# Sort by most fragile first
df_sorted = df.sort_values("multiplier_required", ascending=True)

# Top N most fragile scenarios
topN = 8
most_fragile = df_sorted.dropna(subset=["multiplier_required"]).head(topN).copy()

## Write outputs ##

out_path = os.path.join(OUT_DIR, "reverse_stress_results.csv")
frag_path = os.path.join(OUT_DIR, "reverse_stress_most_fragile.csv")

df_sorted.to_csv(out_path, index=False)
most_fragile.to_csv(frag_path, index=False)

print(f"Wrote {out_path} (rows={len(df_sorted)})")
print(f"Wrote {frag_path} (rows={len(most_fragile)})")

print("\nReverse stress target loss:", TARGET_LOSS)
print("\nMost fragile scenarios (smallest multiplier to hit target):")
cols_show = [c for c in ["scenario", "pnl_total", "target_loss", "multiplier_required", "already_breaches", "type"] if c in most_fragile.columns]
print(most_fragile[cols_show].to_string(index=False))

Wrote data/reverse_stress_results.csv (rows=8)
Wrote data/reverse_stress_most_fragile.csv (rows=5)

Reverse stress target loss: -0.0159339083189366

Most fragile scenarios (smallest multiplier to hit target):
                  scenario  pnl_total  target_loss  multiplier_required  already_breaches        type
          FX_EURUSD_-15pct  -0.037500    -0.015934             0.424904              True risk_factor
            Macro_Combined  -0.018500    -0.015934             0.861292              True       mixed
          Credit_IG_+150bp  -0.018000    -0.015934             0.885217              True risk_factor
Historical_WorstDay_Window  -0.015404    -0.015934             1.034380             False      ticker
          FX_USDJPY_+10pct  -0.010000    -0.015934             1.593391             False risk_factor


## Liquidity

In [14]:
# Liquidity horizons (FRTB bucket assignments, in days)
# Assign per instrument based on asset class
LIQUIDITY_HORIZONS = {
    "IEF":    10,   # rates ETF — FRTB LH bucket 2
    "TLT":    10,   # rates ETF — FRTB LH bucket 2
    "LQD":    20,   # IG credit ETF — FRTB LH bucket 3
    "HYG":    20,   # HY credit ETF — FRTB LH bucket 3
    "EURUSD": 10,   # G10 FX — FRTB LH bucket 2
    "USDJPY": 10,   # G10 FX — FRTB LH bucket 2
}

def compute_liquidity_adjusted_var(
    pos: pd.DataFrame,
    instrument_var: pd.Series,  # index = ticker, value = 1-day VaR contribution
    horizons: dict,
    base_horizon: int = 1,
) -> dict:
    """
    Scale each instrument's VaR contribution by sqrt(LH / base_horizon),
    then re-aggregate. Conservative: no diversification benefit assumed
    across liquidity buckets (sum in quadrature per bucket, then sum buckets).
    """
    results = []
    for _, row in pos.iterrows():
        ticker = row["ticker"]
        lh = horizons.get(ticker, 10)  # default 10d if unknown
        w = row["weight"]
        var_1d = float(instrument_var.get(ticker, 0.0))
        scaled = var_1d * np.sqrt(lh / base_horizon)
        results.append({
            "ticker": ticker,
            "desk": row["desk"],
            "weight": w,
            "liquidity_horizon_days": lh,
            "var_1d": var_1d,
            "var_lh_scaled": scaled,
        })

    df = pd.DataFrame(results)

    # Aggregate: sum of squares across instruments, then sqrt (no cross-instrument netting)
    lavar = float(np.sqrt((df["var_lh_scaled"] ** 2).sum()))
    wavg_lh = float(
        (df["liquidity_horizon_days"] * df["weight"].abs()).sum()
        / df["weight"].abs().sum()
    )

    return {
        "lavar": lavar,
        "wavg_lh": wavg_lh,
        "detail": df,
    }


if __name__ == "__main__":
    DATA_DIR = "data"

    pos = pd.read_csv("positions_stress_ready.csv")

    # Load instrument-level VaR contributions
    # You need a file mapping ticker -> 1-day VaR contribution
    # If you don't have this yet, approximate: weight * portfolio_var / gross_exposure
    risk = pd.read_csv(f"{DATA_DIR}/risk_metrics.csv", parse_dates=["date"])
    risk_total = risk[risk["desk"] == "TOTAL"].sort_values("date")
    latest_var = float(risk_total.iloc[-1]["var_99"])

    gross = pos["weight"].abs().sum()
    instrument_var = pd.Series(
        {row["ticker"]: abs(row["weight"]) / gross * latest_var
         for _, row in pos.iterrows()}
    )

    result = compute_liquidity_adjusted_var(pos, instrument_var, LIQUIDITY_HORIZONS)

    out = pd.DataFrame([{
        "lavar_99": result["lavar"],
        "wavg_liquidity_horizon_days": result["wavg_lh"],
        "base_var_99": latest_var,
        "lavar_multiplier": result["lavar"] / latest_var if latest_var > 0 else np.nan,
    }])

    out.to_csv(f"{DATA_DIR}/liquidity_adjusted_var.csv", index=False)
    result["detail"].to_csv(f"{DATA_DIR}/liquidity_var_detail.csv", index=False)

    print(out.to_string(index=False))
    print("\nInstrument detail:")
    print(result["detail"].to_string(index=False))

 lavar_99  wavg_liquidity_horizon_days  base_var_99  lavar_multiplier
 0.005158                    13.703704     0.003187           1.61865

Instrument detail:
ticker   desk  weight  liquidity_horizon_days   var_1d  var_lh_scaled
   IEF  Rates    0.35                      10 0.000826       0.002613
   TLT  Rates   -0.15                      10 0.000354       0.001120
EURUSD     FX    0.25                      10 0.000590       0.001866
USDJPY     FX   -0.10                      10 0.000236       0.000747
   LQD Credit    0.30                      20 0.000708       0.003167
   HYG Credit   -0.20                      20 0.000472       0.002111


## LaR

In [15]:
DATA_DIR = "data"

LIQUIDITY_HORIZONS = {
    "IEF":    10,
    "TLT":    10,
    "LQD":    20,
    "HYG":    20,
    "EURUSD": 10,
    "USDJPY": 10,
}

def compute_lar(
    pos: pd.DataFrame,
    scen_total: pd.DataFrame,
    scen_contributors: pd.DataFrame,
    horizons: dict,
) -> dict:
    """
    For each scenario, simulate the cumulative P&L path over each instrument's
    liquidity horizon, assuming the daily stress loss repeats each day until
    the position is fully liquidated (linear wind-down). Peak cumulative
    drawdown = margin requirement estimate for that scenario.
    """
    results = []

    for _, scen_row in scen_total.iterrows():
        scenario = scen_row["scenario"]
        total_pnl = float(scen_row["pnl_total"])

        # Get instrument-level stress P&L for this scenario
        inst_contribs = scen_contributors[
            (scen_contributors["scenario"] == scenario) &
            (scen_contributors["contributor_type"] == "INSTRUMENT")
        ].copy()

        if inst_contribs.empty:
            # Fall back: allocate total P&L proportionally by gross weight
            gross = pos["weight"].abs().sum()
            inst_contribs = pd.DataFrame([{
                "contributor": row["ticker"],
                "pnl": total_pnl * abs(row["weight"]) / gross * np.sign(row["weight"]),
            } for _, row in pos.iterrows()])
            inst_contribs["contributor"] = inst_contribs["contributor"].str.split(" | ").str[-1]

        # Simulate daily wind-down for each instrument
        max_horizon = max(horizons.values())
        daily_pnl = np.zeros(max_horizon)

        for _, inst_row in inst_contribs.iterrows():
            # Parse ticker from "Desk | TICKER" format if needed
            raw = str(inst_row["contributor"])
            ticker = raw.split(" | ")[-1].strip() if " | " in raw else raw.strip()

            lh = horizons.get(ticker, 10)
            inst_loss = float(inst_row["pnl"])

            # Linear wind-down: on day t, fraction (lh - t) / lh of position remains
            for t in range(lh):
                remaining = (lh - t) / lh
                daily_pnl[t] += inst_loss * remaining

        # Cumulative path
        cumulative = np.cumsum(daily_pnl)
        peak_drawdown = float(np.min(cumulative))  # most negative point = margin need
        day_of_peak = int(np.argmin(cumulative)) + 1

        results.append({
            "scenario": scenario,
            "stress_pnl_day1": total_pnl,
            "peak_cumulative_loss": peak_drawdown,
            "day_of_peak": day_of_peak,
            "max_horizon_days": max_horizon,
            "margin_estimate": abs(peak_drawdown) if peak_drawdown < 0 else 0.0,
        })

    return pd.DataFrame(results).sort_values("peak_cumulative_loss")


if __name__ == "__main__":
    pos = pd.read_csv("positions_stress_ready.csv")
    scen_total = pd.read_csv(f"{DATA_DIR}/scenario_results_total.csv")
    scen_contributors = pd.read_csv(f"{DATA_DIR}/scenario_top_contributors.csv")

    lar_df = compute_lar(pos, scen_total, scen_contributors, LIQUIDITY_HORIZONS)

    lar_df.to_csv(f"{DATA_DIR}/liquidity_at_risk.csv", index=False)
    print(lar_df.to_string(index=False))

    worst = lar_df.iloc[0]
    print(f"\nWorst case: {worst['scenario']}")
    print(f"  Day-1 stress loss:       {worst['stress_pnl_day1']:.4f}")
    print(f"  Peak cumulative loss:    {worst['peak_cumulative_loss']:.4f}  (day {worst['day_of_peak']})")
    print(f"  Margin estimate:         {worst['margin_estimate']:.4f}")

                  scenario  stress_pnl_day1  peak_cumulative_loss  day_of_peak  max_horizon_days  margin_estimate
          FX_EURUSD_-15pct        -0.037500             -0.206250           10                20         0.206250
          Credit_IG_+150bp        -0.018000             -0.189000           20                20         0.189000
Historical_WorstDay_Window        -0.015404             -0.131331           20                20         0.131331
          FX_USDJPY_+10pct        -0.010000             -0.055000           10                20         0.055000
            Macro_Combined        -0.018500             -0.047200            8                20         0.047200
      Rates_Parallel_200bp         0.008500              0.008500            1                20         0.000000
           Rates_Steepener         0.011000              0.011000            1                20         0.000000
          Credit_HY_+400bp         0.024000              0.024000            1          

### Report

In [ ]:
AS_OF_DATE = None  # None for latest; or '2016-06-24', '2020-03-16', '2023-03-13'

DATA_DIR = "data"
OUT_DIR = "outputs"

# Limits (tuneable)
VAR_LIMIT_TOTAL = 0.30 * capital_with_rnim
VAR_NEAR_LIMIT = 0.80


def fig_to_base64(fig) -> str:
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def ensure_rnim_register(path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df = pd.DataFrame(
        [
            {
                "rnim_item": "Liquidity / gap risk (residual)",
                "why_not_captured": "LA-VaR captures horizon scaling but not market impact, spread widening, or funding liquidity",
                "proxy_metric": "LA-VaR multiplier, stressed ES, LaR margin estimate",
                "current_signal": f"LA-VaR={lavar:.4f} ({lavar_mult:.2f}×base); LaR={worst_lar_margin:.4f}",
                "governance_action": "LA-VaR now in model; residual: add bid-ask spread overlay and funding liquidity reserve buffer",
            },
            {
                "rnim_item": "Correlation breakdown",
                "why_not_captured": "Correlations can change sharply in stress",
                "proxy_metric": "Diversification benefit + concentration metrics",
                "current_signal": "",
                "governance_action": "Monitor diversification + scenario concentration; apply add-on in severe stress",
            },
            {
                "rnim_item": "Jump-to-default / event risk",
                "why_not_captured": "Discrete events may not be represented in history",
                "proxy_metric": "HY exposure share + scenario-based overlays",
                "current_signal": "",
                "governance_action": "Scenario overlays; RNIM add-on; escalation on major events",
            },
            {
                "rnim_item": "Proxy / basis risk",
                "why_not_captured": "ETFs/FX pairs are proxies for risk factors",
                "proxy_metric": "Scenario sensitivity and historical fit checks",
                "current_signal": "",
                "governance_action": "Document limitations; calibrate shocks conservatively; review periodically",
            },
        ]
    )
    df.to_csv(path, index=False)


def html_table(df: pd.DataFrame, max_rows: int = 15) -> str:
    if df is None or df.empty:
        return "<p class='note'>No data available.</p>"
    d = df.head(max_rows).copy()
    return d.to_html(index=False, escape=True, border=0, classes="tbl")


def pick_latest_upto(df: pd.DataFrame, date_col: str, asof: pd.Timestamp) -> pd.Series:
    d = df.sort_values(date_col)
    d = d[d[date_col] <= asof]
    if d.empty:
        raise ValueError(f"No rows on/before {asof.date()} in dataset with date col '{date_col}'.")
    return d.iloc[-1]


def safe_read_csv(path: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **kwargs) if os.path.exists(path) else pd.DataFrame()


## Resolve report as-of date ##
meta_path = os.path.join(DATA_DIR, "asof_metadata.csv")
if not os.path.exists(meta_path):
    raise ValueError("Missing data/asof_metadata.csv. Re-run step2_ingest.py.")
meta = pd.read_csv(meta_path)

true_asof_str = str(meta.loc[0, "true_asof_date"]) if "true_asof_date" in meta.columns else ""
safe_asof_str = str(meta.loc[0, "safe_asof_date"]) if "safe_asof_date" in meta.columns else ""
file_last_str = str(meta.loc[0, "file_last_date"]) if "file_last_date" in meta.columns else ""

if not true_asof_str or true_asof_str.lower() == "nan":
    raise ValueError("true_asof_date is empty in data/asof_metadata.csv (insufficient complete observed data).")

true_asof_date = pd.to_datetime(true_asof_str)
safe_asof_date = pd.to_datetime(safe_asof_str) if safe_asof_str and safe_asof_str.lower() != "nan" else true_asof_date

if AS_OF_DATE is not None:
    report_date = pd.to_datetime(AS_OF_DATE)
    if report_date > true_asof_date:
        raise ValueError(f"Requested as-of {AS_OF_DATE} is after latest complete date {true_asof_str}.")
else:
    report_date = true_asof_date

report_date_str = report_date.strftime("%Y-%m-%d")
report_generated_ts = datetime.now(tz=ZoneInfo("Europe/Amsterdam")).strftime("%Y-%m-%d %H:%M %Z")


## Load datasets (new pipeline paths) ##
risk_metrics = pd.read_csv(os.path.join(DATA_DIR, "risk_metrics.csv"), parse_dates=["date"])
div = pd.read_csv(os.path.join(DATA_DIR, "diversification.csv"), parse_dates=["date"])
stressed = pd.read_csv(os.path.join(DATA_DIR, "stressed_metrics.csv"))  # one row

back = pd.read_csv(os.path.join(DATA_DIR, "backtest_total.csv"), parse_dates=["date"])

# validation flags with alerts fallback
validation_flags = safe_read_csv(os.path.join(DATA_DIR, "validation_flags.csv"), parse_dates=["date"])
alerts_legacy = safe_read_csv(os.path.join(DATA_DIR, "alerts.csv"), parse_dates=["date"])

# Scenario outputs
scen_total = safe_read_csv(os.path.join(DATA_DIR, "scenario_results_total.csv"))
scen_defs = safe_read_csv(os.path.join(DATA_DIR, "scenario_definitions.csv"))
scen_top = safe_read_csv(os.path.join(DATA_DIR, "scenario_top_contributors.csv"))
scen_conc = safe_read_csv(os.path.join(DATA_DIR, "scenario_concentration_metrics.csv"))

# Portfolio change attribution outputs
chg_total = safe_read_csv(os.path.join(DATA_DIR, "stress_change_attribution_total.csv"))
chg_desks = safe_read_csv(os.path.join(DATA_DIR, "stress_change_attribution_desk.csv"))
chg_factors = safe_read_csv(os.path.join(DATA_DIR, "stress_change_attribution_factor.csv"))

# Liquidity
liq = safe_read_csv(os.path.join(DATA_DIR, "liquidity_adjusted_var.csv"))

lavar = float(liq.iloc[0]["lavar_99"]) if not liq.empty else float("nan")
wavg_lh = float(liq.iloc[0]["wavg_liquidity_horizon_days"]) if not liq.empty else float("nan")
lavar_mult = float(liq.iloc[0]["lavar_multiplier"]) if not liq.empty else float("nan")

lar = safe_read_csv(os.path.join(DATA_DIR, "liquidity_at_risk.csv"))

worst_lar_scenario = lar.iloc[0]["scenario"] if not lar.empty else "N/A"
worst_lar_margin   = float(lar.iloc[0]["margin_estimate"]) if not lar.empty else float("nan")
worst_lar_day      = int(lar.iloc[0]["day_of_peak"]) if not lar.empty else 0

lar_html = html_table(lar[["scenario","stress_pnl_day1","peak_cumulative_loss",
                             "day_of_peak","margin_estimate"]], max_rows=10) \
           if not lar.empty else "<p class='note'>LaR not computed.</p>"

# RNIM quantified overlay
cap_rnim = safe_read_csv(os.path.join(DATA_DIR, "capital_with_rnim.csv"))

# RNIM register (qualitative)
rnim_path = os.path.join(DATA_DIR, "rnim_register.csv")
ensure_rnim_register(rnim_path)
rnim = pd.read_csv(rnim_path)

if not liq.empty:
    mask = rnim["rnim_item"] == "Liquidity / gap risk"
    rnim.loc[mask, "current_signal"] = (
        f"LA-VaR={lavar:.4f} ({lavar_mult:.2f}×base); wavg LH={wavg_lh:.1f}d"
    )
  



## Core KPIs as-of ##
risk_total = risk_metrics[risk_metrics["desk"] == "TOTAL"].sort_values("date")
risk_total_upto = risk_total[risk_total["date"] <= report_date]
if risk_total_upto.empty:
    raise ValueError(f"No TOTAL risk metric rows on/before {report_date_str}")

latest_row = risk_total_upto.iloc[-1]
latest_date_for_tables = pd.to_datetime(latest_row["date"])

latest_var = float(latest_row.get("var_99", np.nan))
latest_es = float(latest_row.get("es_975", np.nan))  # FRTB-style naming

# Limit status
var_util = latest_var / VAR_LIMIT_TOTAL if (np.isfinite(latest_var) and VAR_LIMIT_TOTAL > 0) else float("nan")
if np.isfinite(var_util) and var_util >= 1.0:
    var_limit_status, limit_class = "BREACH", "status-breach"
elif np.isfinite(var_util) and var_util >= VAR_NEAR_LIMIT:
    var_limit_status, limit_class = "NEAR LIMIT", "status-warn"
else:
    var_limit_status, limit_class = "OK", "status-ok"

# Diversification as-of 
div_upto = div[div["date"] <= report_date].sort_values("date")
if div_upto.empty:
    raise ValueError(f"No diversification rows on/before {report_date_str}")
div_last = div_upto.iloc[-1]

div_benefit_var = float(div_last.get("diversification_benefit_var", np.nan))
div_benefit_es = float(div_last.get("diversification_benefit_es", np.nan))
sum_desk_var = float(div_last.get("sum_desk_var", np.nan))
total_var = float(div_last.get("total_var", np.nan))
sum_desk_es = float(div_last.get("sum_desk_es", np.nan))
total_es = float(div_last.get("total_es", np.nan))

# Desk risk table at latest available date <= asof
latest_desk = (
    risk_metrics[risk_metrics["date"] == latest_date_for_tables]
    .query("desk != 'TOTAL'")
    .sort_values("var_99", ascending=False)
    .rename(columns={"var_99": "VaR_99", "es_975": "ES_97_5"})
)

# Stressed calibration (static)
svar_total = float(stressed.iloc[0].get("svar_99", np.nan))
ses_total = float(stressed.iloc[0].get("ses_97_5", stressed.iloc[0].get("ses_97_5", np.nan)))

svar_ratio = svar_total / latest_var if (np.isfinite(svar_total) and np.isfinite(latest_var) and latest_var > 0) else float("nan")

# Capital proxies
capital_frtb_model = 1.5 * max(latest_es, ses_total) if (np.isfinite(latest_es) and np.isfinite(ses_total)) else float("nan")

# RNIM quantified capital
capital_with_rnim = float("nan")
total_rnim_addon = float("nan")
if not cap_rnim.empty:
    capital_with_rnim = float(cap_rnim.iloc[0].get("capital_with_rnim", np.nan))
    total_rnim_addon = float(cap_rnim.iloc[0].get("total_rnim_addon", np.nan))

# Backtesting as-of
back_upto = back[back["date"] <= report_date].sort_values("date")
if back_upto.empty:
    raise ValueError(f"No backtest rows on/before {report_date_str}")
back_latest = back_upto.iloc[-1]
exc_today = int(back_latest.get("exception", 0))
exc_250 = int(back_latest["exceptions_rolling"]) if pd.notna(back_latest.get("exceptions_rolling", np.nan)) else None
kupiec_p = float(back_latest["kupiec_pvalue"]) if pd.notna(back_latest.get("kupiec_pvalue", np.nan)) else float("nan")

if exc_today == 0:
    exception_text, exception_class = "No exception", "status-ok"
else:
    exception_text, exception_class = "Exception", "status-breach"

# Validation flags with alerts fallback
recent_flags_html = "<p class='note'>No validation flags up to the as-of date.</p>"
if not validation_flags.empty:
    flags_upto = validation_flags[validation_flags["date"] <= report_date].sort_values("date").tail(10).copy()
    if not flags_upto.empty:
        flags_upto["date"] = flags_upto["date"].dt.strftime("%Y-%m-%d")
        recent_flags_html = html_table(flags_upto, max_rows=10)
elif not alerts_legacy.empty:
    alerts_upto = alerts_legacy[alerts_legacy["date"] <= report_date].sort_values("date").tail(10).copy()
    if not alerts_upto.empty:
        alerts_upto["date"] = alerts_upto["date"].dt.strftime("%Y-%m-%d")
        recent_flags_html = html_table(alerts_upto, max_rows=10)

# Scenario summary tables
scenario_summary_html = "<p class='note'>Scenario stress outputs not found (run scenario stress engine).</p>"
scenario_top_html = "<p class='note'>Scenario attribution not found.</p>"
scenario_conc_html = "<p class='note'>Scenario concentration metrics not found.</p>"

if not scen_total.empty:
    scen_sum = scen_total.copy()

    # Merge in scenario type/notes/etc
    if not scen_defs.empty and "scenario" in scen_defs.columns:
        scen_sum = scen_sum.merge(
            scen_defs[["scenario", "type", "notes"]].copy(),
            on="scenario",
            how="left",
        )

    # Merge in worst-rank (computed in concentration metrics script)
    if not scen_conc.empty and "scenario" in scen_conc.columns and "worst_rank_by_total_loss" in scen_conc.columns:
        scen_sum = scen_sum.merge(
            scen_conc[["scenario", "worst_rank_by_total_loss"]].copy(),
            on="scenario",
            how="left",
        )

    if "worst_rank_by_total_loss" not in scen_sum.columns or scen_sum["worst_rank_by_total_loss"].isna().all():
        scen_sum = scen_sum.sort_values("pnl_total").reset_index(drop=True)
        scen_sum["worst_rank_by_total_loss"] = np.arange(1, len(scen_sum) + 1)

    # Sort worst to best
    scen_sum = scen_sum.sort_values("pnl_total").head(10)

    # Keep only clean columns
    cols_keep = ["worst_rank_by_total_loss", "scenario", "type", "notes", "pnl_total"]
    cols_keep = [c for c in cols_keep if c in scen_sum.columns]
    scen_sum = scen_sum[cols_keep].copy()

    scen_sum = scen_sum.rename(
        columns={
            "worst_rank_by_total_loss": "Rank",
            "scenario": "Scenario",
            "type": "Type",
            "notes": "Notes",
            "pnl_total": "Total Stress P&L"
        }
    )

    scenario_summary_html = html_table(scen_sum, max_rows=10)

if not scen_top.empty:
    scen_rank=2 #default = 1
    # Show top contributors for worst scenario (by total loss), else first scenario
    worst_scen = None
    if not scen_total.empty and "pnl_total" in scen_total.columns:
        worst_scen = scen_total.sort_values("pnl_total").iloc[scen_rank-1]["scenario"]
    else:
        worst_scen = scen_top["scenario"].iloc[scen_rank-1]

    top_subset = scen_top[scen_top["scenario"] == worst_scen].copy()
    # Keep only the most interpretable columns
    cols = [c for c in ["scenario", "contributor_type", "rank", "contributor", "pnl", "abs_pnl"] if c in top_subset.columns]
    scenario_top_html = html_table(top_subset[cols], max_rows=15)

if not scen_conc.empty:
    # Show worst 5 scenarios concentration snapshot
    conc_subset = scen_conc.copy()
    if "pnl_total" in conc_subset.columns:
        conc_subset = conc_subset.sort_values("pnl_total").head(8)
    cols_keep = ["scenario","pnl_total",
                 "desk_top1_share_abs","desk_hhi_abs",
                 "factor_top1_share_abs","factor_top3_share_abs","factor_hhi_abs",
                 "instrument_top1_share_abs","instrument_top3_share_abs","instrument_hhi_abs"]

    conc_subset = conc_subset[cols_keep]
    conc_subset = conc_subset.rename(
        columns={"scenario":"Scenario","pnl_total":"TotalP&L",
                 "desk_top1_share_abs":"DeskTop1","desk_hhi_abs":"DeskHHI",
                 "factor_top1_share_abs":"FacTop1","factor_top3_share_abs":"FacTop3","factor_hhi_abs":"FacHHI",
                 "instrument_top1_share_abs":"InstTop1","instrument_top3_share_abs":"InstTop3","instrument_hhi_abs":"InstHHI"})
    scenario_conc_html = html_table(conc_subset, max_rows=10)

# Reverse Stress Testing

rev_frag = safe_read_csv(os.path.join(DATA_DIR, "reverse_stress_most_fragile.csv"))
rev_all  = safe_read_csv(os.path.join(DATA_DIR, "reverse_stress_results.csv"))

reverse_stress_html = "<p class='note'>Reverse stress testing outputs not found (run step10_reverse_stress_testing.py).</p>"

if not rev_frag.empty:
    cols_keep = [
        "scenario",
        "pnl_total",
        "target_loss",
        "multiplier_required",
        "already_breaches",
        "type",
    ]
    cols_keep = [c for c in cols_keep if c in rev_frag.columns]
    t = rev_frag[cols_keep].copy()

    # More readable
    if "already_breaches" in t.columns:
        t["already_breaches"] = t["already_breaches"].astype(bool)

    reverse_stress_html = html_table(t, max_rows=10)

# Portfolio change attribution tables
chg_summary_html = "<p class='note'>Portfolio change attribution not found (run step8_portfolio_change_attribution).</p>"
chg_desk_html = "<p class='note'>No desk-level change attribution.</p>"
chg_factor_html = "<p class='note'>No factor-level change attribution.</p>"

if not chg_total.empty:
    chg_total2 = chg_total.copy()
    # show top worsening scenarios by delta_stress
    if "delta_stress" in chg_total2.columns:
        chg_total2 = chg_total2.sort_values("delta_stress").head(8)
    chg_summary_html = html_table(chg_total2, max_rows=10)

if not chg_desks.empty:
    worst_scen2 = None
    if not chg_total.empty and "delta_stress" in chg_total.columns:
        worst_scen2 = chg_total.sort_values("delta_stress").iloc[0]["scenario"]
        g = chg_desks[chg_desks["scenario"] == worst_scen2].sort_values("delta_stress").head(10)
        chg_desk_html = html_table(g, max_rows=10)

if not chg_factors.empty:
    if not chg_total.empty and "delta_stress" in chg_total.columns:
        worst_scen2 = chg_total.sort_values("delta_stress").iloc[0]["scenario"]
        g = chg_factors[chg_factors["scenario"] == worst_scen2].sort_values("delta_stress").head(10)
        chg_factor_html = html_table(g, max_rows=10)


## Charts (up to as-of date) ##
fig1 = plt.figure(figsize=(5, 3))
plt.plot(risk_total_upto["date"], risk_total_upto["var_99"])
plt.title("TOTAL Historical VaR (99%)")
plt.xlabel("Date")
plt.ylabel("VaR (loss units)")
img_var = fig_to_base64(fig1)

"""fig1b = plt.figure()
plt.plot(risk_total_upto["date"], risk_total_upto["es_975"])
plt.title("TOTAL Historical ES (97.5%)")
plt.xlabel("Date")
plt.ylabel("ES (loss units)")
img_es = fig_to_base64(fig1b)"""

fig2 = plt.figure(figsize=(5, 3))
plt.plot(div_upto["date"], div_upto.get("diversification_benefit_var", div_upto.get("diversification_benefit", np.nan)))
plt.title("Diversification Benefit (VaR)")
plt.xlabel("Date")
plt.ylabel("Benefit (loss units)")
img_div = fig_to_base64(fig2)

fig3 = plt.figure(figsize=(5, 3))
plt.plot(back_upto["date"], back_upto["exceptions_rolling"])
plt.title("Rolling Exceptions (last 250d)")
plt.xlabel("Date")
plt.ylabel("Exceptions (count)")
img_exc = fig_to_base64(fig3)


## HTML/CSS ##
css = """
<style>
  body { font-family: Arial, sans-serif; margin: 24px; color: #111; }
  h1 { margin-bottom: 6px; }
  h2 { margin-top: 22px; }
  .sub { color: #444; margin-top: 0; line-height: 1.4; }
  .card { border: 1px solid #ddd; border-radius: 10px; padding: 14px; }
  .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
  .note { font-size: 12px; color: #555; }
  img { width: 100%; height: auto; }

  .tbl { border-collapse: collapse; width: 100%; font-size: 13px; }
  .tbl th, .tbl td { border-bottom: 1px solid #eee; padding: 8px; text-align: left; vertical-align: top; }
  .tbl th { background: #fafafa; }

  .kpi { display: grid; grid-template-columns: repeat(5, 1fr); gap: 12px; }
  @media (max-width: 1100px) { .kpi { grid-template-columns: repeat(3, 1fr); } }
  @media (max-width: 700px)  { .kpi { grid-template-columns: repeat(2, 1fr); } }
  
  .kpi .box { border: 1px solid #eee; border-radius: 10px; padding: 10px; }
  .kpi .label { font-size: 12px; color: #555; }
  .kpi .value { font-size: 20px; font-weight: 700; margin-top: 6px; }

  .status-ok { color: #1a7f37; font-weight: 700; }
  .status-warn { color: #b36b00; font-weight: 700; }
  .status-breach { color: #b42318; font-weight: 700; }

  .grid-3 {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 16px;
}

@media (max-width: 1000px) {
  .grid-3 {
    grid-template-columns: 1fr;
  }
}
</style>
"""

html = f"""
<html>
  <head>
    <meta charset="utf-8"/>
    <title>Market Risk Report — {report_date_str}</title>
    {css}
  </head>
  <body>
    <h1>Market Risk Report</h1>
    <p class="sub">
      <b>Report AS-OF:</b> {report_date_str}
      &nbsp;|&nbsp;
      <b>Market data AS-OF:</b> {true_asof_str} (safe: {safe_asof_date.strftime("%Y-%m-%d")})
      &nbsp;|&nbsp;
      <b>Report generated:</b> {report_generated_ts}
      <br/>
      Consolidated Trading Book (Toy) — Rates / FX / Credit
    </p>

    <div class="card">
  <div class="kpi">

    <div class="box">
      <div class="label">TOTAL VaR (99%)</div>
      <div class="value">{latest_var:.6f}</div>
    </div>

    <div class="box">
      <div class="label">Stressed VaR (99%)</div>
      <div class="value">{svar_total:.6f}</div>
      <div class="note">SVaR / VaR: {svar_ratio:.2f}×</div>
    </div>

    <div class="box">
      <div class="label">Liquidity-Adjusted VaR (99%)</div>
      <div class="value">{lavar:.6f}</div>
      <div class="note">FRTB-style LH scaling · {lavar_mult:.2f}× base VaR · wavg LH {wavg_lh:.1f}d</div>
    </div>

    <div class="box">
      <div class="label">Model Capital (Proxy)</div>
      <div class="value">{capital_frtb_model:.6f}</div>
      <div class="note">1.5 × max(ES, Stressed ES)</div>
    </div>

    <div class="box">
      <div class="label">RNIM Add-on (Proxy)</div>
      <div class="value">{total_rnim_addon:.6f}</div>
      <div class="note">Liquidity + concentration + JTD + correlation overlays</div>
    </div>

    <div class="box">
      <div class="label">TOTAL ES (97.5%)</div>
      <div class="value">{latest_es:.6f}</div>
    </div>

    <div class="box">
      <div class="label">Stressed ES (97.5%)</div>
      <div class="value">{ses_total:.6f}</div>
      <div class="note">Fixed stress window calibration</div>
    </div>

    <div class="box">
      <div class="label">Liquidity-at-Risk (worst scenario)</div>
      <div class="value">{worst_lar_margin:.4f}</div>
      <div class="note">{worst_lar_scenario} · peak day {worst_lar_day}</div>
    </div>

    <div class="box">
      <div class="label">Capital incl. RNIM</div>
      <div class="value">{capital_with_rnim:.6f}</div>
    </div>

    <div class="box">
      <div class="label">VaR Limit Utilization</div>
      <div class="value">{var_util:.1%}</div>
      <div class="note">
        Limit: {VAR_LIMIT_TOTAL:.6f} (0.3 × Capital) •
        Status: <span class="{limit_class}"><b>{var_limit_status}</b></span>
      </div>
    </div>

  </div>

  <p class="note" style="margin-top:10px;">
    Backtesting:
    <span class="{exception_class}">{exception_text}</span>
    &nbsp;•&nbsp; Exceptions (250d): <b>{exc_250}</b>
    &nbsp;•&nbsp; Kupiec p-value: <b>{kupiec_p:.3f}</b>
  </p>

      <p class="note">
        Diversification Benefit (VaR): <b>{div_benefit_var:.6f}</b> (sum desks {sum_desk_var:.6f} − total {total_var:.6f})<br/>
        Diversification Benefit (ES): <b>{div_benefit_es:.6f}</b> (sum desks {sum_desk_es:.6f} − total {total_es:.6f})
      </p>
    </div>

    <div class="grid-3" style="margin-top:16px;">
      <div class="card">
        <h3>VaR Trend</h3>
        <img src="data:image/png;base64,{img_var}"/>
      </div>
    
      <div class="card">
        <h3>Diversification Trend</h3>
        <img src="data:image/png;base64,{img_div}"/>
      </div>
    
      <div class="card">
        <h3>Rolling Exceptions</h3>
        <img src="data:image/png;base64,{img_exc}"/>
      </div>
    </div>

    <div class="card" style="margin-top:16px;">
      <h3>Desk Risk Snapshot — Latest Available</h3>
      {html_table(latest_desk[["desk","VaR_99","ES_97_5"]] if not latest_desk.empty else latest_desk, max_rows=20)}
      <p class="note">Desk breakdown uses the latest risk metrics date ≤ as-of.</p>
    </div>

    <h2>Scenario Stress Testing</h2>

    <div class="card">
      <h3>Worst Scenarios (Total P&amp;L Impact)</h3>
      {scenario_summary_html}
      <p class="note">
        Scenarios are explicit shocks (return-space proxies). Negative values indicate losses for the consolidated book.
      </p>
    </div>

    <div class="grid" style="margin-top:16px;">
      <div class="card">
        <h3>Top Stress Contributors (#{scen_rank} Worst Scenario)</h3>
        {scenario_top_html}
        <p class="note">Breakdown shown for the #{scen_rank} worst scenario by total loss.</p>
        <p class="note">We don't use #1 worst scenario since it is a single factor shock.</p>
      </div>
      <div class="card">
        <h3>Concentration Metrics (Worst Scenarios)</h3>
        {scenario_conc_html}
        <p class="note">Shares are computed on absolute P&amp;L contributions (more concentrated = higher top-share / HHI).</p>
        <p class="note">All desk, factor and instrument variables above are absolute values.</p>
      </div>
    </div>

    <div class="card" style="margin-top:16px;">
      <h3>Reverse Stress Testing — Minimum Shock Multiplier to Hit Loss Threshold</h3>
      {reverse_stress_html}
      <p class="note">
        Reverse stress testing scales each scenario linearly to find the multiplier required to reach the target loss threshold.
        <br/>
        <b>multiplier_required</b> &lt; 1 means the base scenario already breaches the threshold; larger values imply more buffer.
      </p>
    </div>

    <div class="card" style="margin-top:16px;">
      <h3>Liquidity-at-Risk — Margin Requirement Estimates by Scenario</h3>
      {lar_html}
      <p class="note">
        Peak cumulative loss over the instrument liquidation horizon under each stress scenario,
        assuming linear position wind-down. Margin estimate = abs(peak cumulative loss) where negative.
      </p>
    </div>

    <h2>Portfolio Change Attribution</h2>
    <p class="sub">
      T0 is the baseline portfolio
      <br/>
      T1 reflects manual changes: higher IEF exposure, reduced HYG hedge, and slightly increased EURUSD
    </p>
    <div class="card">
      <h3>Stress Change Summary (T1 vs T0)</h3>
      {chg_summary_html}
      <p class="note">
        ΔStress is attributed to position changes only (scenario shocks held constant).
      </p>
    </div>

    <div class="grid" style="margin-top:16px;">
      <div class="card">
        <h3>Top Desk Movers (Worst ΔStress Scenario)</h3>
        {chg_desk_html}
      </div>
      <div class="card">
        <h3>Top Risk Factor Movers (Worst ΔStress Scenario)</h3>
        {chg_factor_html}
      </div>
    </div>

    <h2>Validation &amp; Data Quality</h2>
    <div class="card">
      <h3>Validation Flags (Tail events / anomalies) — last 10</h3>
      {recent_flags_html}
      <p class="note">
        Flags highlight extreme P&amp;L tail events (rolling regime tails) and help validate stress calibration and proxy behavior.
      </p>
    </div>

    <h2>RNIM</h2>
    <div class="card">
      <h3>Risks Not In Model — Monitoring Register</h3>
      {html_table(rnim, max_rows=25)}
      <p class="note">
        RNIM provides governance context for limitations (liquidity, correlation, JTD, proxy/basis risk) and overlay actions.
      </p>
    </div>
  </body>
</html>
"""

os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, f"MarketRiskStressReport_{report_date_str}.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Wrote {out_path}")

Wrote outputs/MarketRiskStressReport_2026-05-29.html
